# 머신러닝 파일별 3 · 3-1 전처리 · 인코딩 · 스케일링 · 정규화 · 누수

**한 줄 요약**

- 글자를 숫자로(인코딩), 크기를 맞추고(스케일링), 길이를 1로(정규화), 시험지 정보는 쓰지 않기(누수 · Pipeline)
- 데이터를 그대로 넣으면 안 되는 이유를 실행으로 확인

- **원본** · `6.머닝러신/03_전처리_평가지표_Titanic/3-1_전처리_05_preprocess.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-2 (전처리 · 누수 · 파이프라인) 1 ~ 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 명목형은 원-핫, 순서형은 Ordinal, LabelEncoder는 주로 y라는 선택 기준을 말한다
2. 레이블 인코딩이 선형 모델에 가짜 순서를 넣는 것을 확률 세 줄로 보인다
3. Standard · MinMax · Robust 스케일러의 차이와 정규화(행 기준)와의 차이를 설명한다
4. `fit`은 train에만, `transform`은 train · test 둘 다라는 두 규칙을 지킨다
5. 누수는 점수가 아니라 절차로 판단한다는 것을 난수 데이터 0.870 vs 0.520으로 설명한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">성별 · 항구처럼 글자인 열이 있을 때</td>
<td style="text-align: left;">원-핫 · Ordinal (1장)</td>
</tr>
<tr>
<td style="text-align: left;">리뷰 · 문장을 모델에 넣을 때</td>
<td style="text-align: left;">CountVectorizer 사전 (2장)</td>
</tr>
<tr>
<td style="text-align: left;">KNN · 로지스틱 성능이 이상하게 낮을 때</td>
<td style="text-align: left;">단위 차이 → 스케일링 (3장 · 종합 실습)</td>
</tr>
<tr>
<td style="text-align: left;">문장 · 임베딩이 비슷한지 비교할 때</td>
<td style="text-align: left;">정규화 + 코사인 유사도 (4장)</td>
</tr>
<tr>
<td style="text-align: left;">교차검증 점수가 너무 좋을 때</td>
<td style="text-align: left;">Pipeline 밖에서 fit한 전처리가 있는지 (5장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfm3_map.png" alt="3-1 전처리 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (전체 실행 30초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">인코딩</td>
<td style="text-align: left;">글자는 어떻게 숫자로 바꾸고, 번호만 붙이면 무엇이 문제인가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">텍스트 인코딩</td>
<td style="text-align: left;">문장은 어떻게 숫자가 되고, 출력이 나왔다고 근거가 있는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">스케일링</td>
<td style="text-align: left;">값의 크기 차이는 왜 문제이고, 세 스케일러는 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">정규화</td>
<td style="text-align: left;">정규화는 스케일링과 무엇이 다르고, 왜 코사인 유사도와 짝을 이루는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">데이터 누수</td>
<td style="text-align: left;">scaler를 맞출 때 test까지 보면 왜 문제이고, Pipeline은 무엇을 막아 주는가?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">2회차 펭귄 KNN 0.7801은 스케일링 Pipeline으로 얼마나 고쳐지는가?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

1 ~ 5장은 원본처럼 작은 예제 · Iris · 난수를 쓰고, 6장 종합 실습만 원본 폴더의 펭귄 CSV를 읽음. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">numpy · pandas · scikit-learn</td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;"><code>np.random.seed(42)</code> · <code>random_state=42</code> (원본과 같음)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">작은 표 · 문장 6개 · Iris · 난수 100 × 5000 · 펭귄 342마리(6장)</td>
</tr>
<tr>
<td style="text-align: left;">원본 그림</td>
<td style="text-align: left;">셰프 비유 스크린샷 2장 → 0장 힌트의 글로 옮김</td>
</tr>
</tbody>
</table>

In [1]:
# 준비: 이 노트북 전체에서 쓰는 라이브러리 (모델 · 전처리는 섹션마다 불러옴)
import numpy as np   # 배열 · 난수
import pandas as pd  # 표
import sklearn       # 버전 확인

print("sklearn:", sklearn.__version__)  # 원본은 1.6.1 · 다르면 끝자리가 다를 수 있음
print("pandas:", pd.__version__)        # 표 출력 모양 확인

sklearn: 1.9.0
pandas: 3.0.3


### 원본 목표 · 전처리가 필요한 이유

- **핵심 질문** · 데이터를 그대로 넣으면 안 되는 이유는?
- 2회차에서 모델 고르는 기준을 배웠지만, 좋은 모델도 **재료가 준비되지 않으면** 성능이 안 나옴

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">결과</th>
<th style="text-align: left;">해결책</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">문자열 데이터</td>
<td style="text-align: left;">모델이 계산 불가</td>
<td style="text-align: left;"><strong>인코딩</strong> (1 · 2장)</td>
</tr>
<tr>
<td style="text-align: left;">값의 크기 차이(단위 · 범위)</td>
<td style="text-align: left;">거리 기반 모델에서 큰 단위 특성이 거리를 지배</td>
<td style="text-align: left;"><strong>스케일링</strong> (3장)</td>
</tr>
<tr>
<td style="text-align: left;">길이보다 방향이 중요</td>
<td style="text-align: left;">의미 · 패턴 비교가 어려움</td>
<td style="text-align: left;"><strong>정규화</strong> (4장)</td>
</tr>
<tr>
<td style="text-align: left;">전처리를 전체 데이터로 fit</td>
<td style="text-align: left;">평가가 무효 (누수)</td>
<td style="text-align: left;"><strong>Pipeline</strong> (5장)</td>
</tr>
</tbody>
</table>

**원본 힌트**

- 원본 그림 1 · 최고의 셰프도 상한 재료로는 요리할 수 없음
- 원본 그림 2 · 모델을 위한 재료 손질 = 전처리
- 3회차 핵심 문장 · 모델은 숫자만 이해하고, **모델에 따라** 값의 크기 차이가 학습에 영향을 줄 수 있음

---
## 1. 인코딩

> **이 장의 흐름**
>
> - Label과 One-Hot으로 같은 색깔 열을 바꿔 봄 → 선형 모델에 넣어 차이 확인
> - 모델을 열어 가짜 순서를 확인 → 트리 · Ordinal로 넓힘

<img src="images/pfm3_encode.png" alt="Label은 순서를 만들고, One-Hot은 있다 · 없다만" width="763">

**파이프라인 위치** · ③ 전처리 — 범주형 열

### 1-1. `A` Label vs One-Hot · `LabelEncoder` · `OneHotEncoder` <sub>원본 셀 2 ~ 7</sub>

> 🎯 **같은 색깔 열을 Label과 One-Hot으로 바꾸면 각각 어떻게 생기는가?**

💡 **쉽게 말하면**

- 모델은 글자를 계산 못 함 → **숫자로 번역**해야 함.
- Label은 번호표, One-Hot은 '이 색인가?' 체크 칸.

**실행 흐름**

1. 색깔 6개 표 만들기
2. Label로 번호 붙이기
3. One-Hot으로 열 3개 만들기
4. 옆으로 합치기

**먼저 예측** · red · green · blue에 붙는 Label 번호는? (알파벳 순일까?) — 답은 코드 실행 결과 아래에

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # 글자 → 숫자

data = pd.DataFrame({"color": ["red", "green", "blue", "red", "green", "blue"]})  # 글자 열 1개
le = LabelEncoder()                                            # 번호 붙이기
data["color_label"] = le.fit_transform(data["color"])          # 0 · 1 · 2 번호 (라이브러리가 정함)
ohe = OneHotEncoder(sparse_output=False)                       # 배열로 받기
ohe_matrix = ohe.fit_transform(data[["color"]])                # 카테고리마다 새 열 · 행 수는 그대로
ohe_df = pd.DataFrame(ohe_matrix, columns=ohe.categories_[0])  # 열 이름 = 카테고리
data_ohe = pd.concat([data, ohe_df], axis=1)                   # 옆으로 합치기
print(data_ohe.to_string())                                    # 원본 셀 6의 표
# 열 개수 = 차원
print("Label 순서:", list(le.classes_), "→ 차원 1 → One-Hot 차원", ohe_matrix.shape[1])

   color  color_label  blue  green  red
0    red            2   0.0    0.0  1.0
1  green            1   0.0    1.0  0.0
2   blue            0   1.0    0.0  0.0
3    red            2   0.0    0.0  1.0
4  green            1   0.0    1.0  0.0
5   blue            0   1.0    0.0  0.0
Label 순서: ['blue', 'green', 'red'] → 차원 1 → One-Hot 차원 3


**결과** · 예측 정답은 알파벳 순 · blue 0 · green 1 · red 2

- `0    red            2   0.0    0.0  1.0` — red = 번호 2 · One-Hot [0, 0, 1]
- `2   blue            0   1.0    0.0  0.0` — blue = 번호 0 · One-Hot [1, 0, 0]
- `Label 순서: ['blue', 'green', 'red'] → 차원 1 → One-Hot 차원 3` — 번호는 알파벳 순 · One-Hot은 열(차원)이 1 → 3

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code></td>
<td style="text-align: left;">0 · 1 · 2 번호 → 순서가 생김</td>
<td style="text-align: left;">주로 <strong>타깃 y</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>OneHotEncoder</code></td>
<td style="text-align: left;">카테고리마다 0 / 1 열 · 순서 없음</td>
<td style="text-align: left;"><strong>순서 없는 X의 출발점</strong></td>
</tr>
<tr>
<td style="text-align: left;">차원</td>
<td style="text-align: left;">모델이 참고하는 입력 열 개수</td>
<td style="text-align: left;">원-핫이면 카테고리 수만큼 늘어남</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 차원이 늘수록 데이터 구조가 복잡해지고 중요하지 않은 정보가 섞일 수 있음
> - 그래서 나중에 차원을 줄이거나 중요한 축만 남기는 방법도 배움

> ✅ **한 줄 정리** · Label은 0 · 1 · 2 번호를, One-Hot은 카테고리마다 0 / 1 열을 만들며 원-핫은 차원을 늘리는 대신 순서를 만들지 않는다.

**딥러닝 코드에서는**

- `nn.Embedding(카테고리 수, 차원)` — 범주가 많으면 원-핫 대신 학습되는 벡터
- `F.one_hot(labels, 10)` — 정답을 원-핫으로

### 1-2. `A` 레이블 인코딩의 함정 · `LogisticRegression` · `y.mean()` <sub>원본 셀 8 ~ 11</sub>

> 🎯 **red · green · blue에 번호만 붙여 선형 모델에 넣으면 모델이 이름표로 이해하는가?**

💡 **쉽게 말하면**

- red · blue면 1, green이면 0 → 사람에게는 쉬운 규칙.
- 그런데 번호 0(blue) · 1(green) · 2(red)는 green이 **가운데**라 직선 하나로 못 가름.

**실행 흐름**

1. 색 100개 · 규칙으로 정답 만들기
2. Label 열 1개 · One-Hot 열 3개
3. 같은 로지스틱으로 학습 정확도
4. 다수 클래스 비율과 비교

**먼저 예측** · Label 방식의 학습 데이터 정확도는 One-Hot과 같을까? — 답은 코드 실행 결과 아래에

In [3]:
from sklearn.linear_model import LogisticRegression  # 선형 모델
from sklearn.metrics import accuracy_score           # 정확도

np.random.seed(42)                                                               # 원본과 같은 난수
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))         # 색 100개
target = colors.map({"red": 1, "blue": 1, "green": 0})  # red · blue → 1, green → 0
df_test = pd.DataFrame({"color": colors, "target": target})                      # 실험용 표
X_label = LabelEncoder().fit_transform(df_test["color"]).reshape(-1, 1)          # 번호 열 1개
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(df_test[["color"]])  # 0 / 1 열 3개
y = df_test["target"]                                                            # 정답
# 같은 모델 · 표현만 다름
acc_label = accuracy_score(y, LogisticRegression().fit(X_label, y).predict(X_label))
acc_onehot = accuracy_score(y, LogisticRegression().fit(X_onehot, y).predict(X_onehot))  # 같은 모델
print("=== 로지스틱 회귀 (선형 모델) ===")                                       # 원본 출력 형식
print(f"레이블 방식 학습 데이터 정확도: {acc_label:.3f}")                        # 번호
print(f"원-핫 방식 학습 데이터 정확도: {acc_onehot:.3f}")                        # 원-핫
print(f"참고 - 다수 클래스 비율: {y.mean():.3f}")                                # 1의 비율

=== 로지스틱 회귀 (선형 모델) ===
레이블 방식 학습 데이터 정확도: 0.640
원-핫 방식 학습 데이터 정확도: 1.000
참고 - 다수 클래스 비율: 0.640


**결과** · 예측 정답은 다름 — Label 0.640 · One-Hot 1.000

- `레이블 방식 학습 데이터 정확도: 0.640` — 다수 클래스 비율과 **정확히 같음** → 모든 입력에 같은 답
- `원-핫 방식 학습 데이터 정확도: 1.000` — 열마다 따로 가중치 → 규칙을 그대로 표현
- `참고 - 다수 클래스 비율: 0.640` — 전부 1로 찍은 점수

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Label + 선형 모델</td>
<td style="text-align: left;">green이 가운데 → 직선 하나로 못 가름</td>
<td style="text-align: left;">쓰지 않기 (명목형 X)</td>
</tr>
<tr>
<td style="text-align: left;">One-Hot + 선형 모델</td>
<td style="text-align: left;">색마다 가중치 하나씩</td>
<td style="text-align: left;">명목형 X의 기본</td>
</tr>
<tr>
<td style="text-align: left;">학습 데이터 정확도</td>
<td style="text-align: left;">표현할 수 있는가만 봄 · 일반화 성능 아님</td>
<td style="text-align: left;">표현력 확인용 (성능은 3-2회차)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이 숫자는 **학습 데이터 정확도** → 일반화 성능이라고 부르면 안 됨 (3-2회차)
> - 지금 보는 것은 같은 데이터를 다르게 표현했을 때 패턴을 표현할 수 있는가

> ✅ **한 줄 정리** · 명목형 X에 번호만 붙이면 선형 모델은 그 번호를 크기로 써서, 쉬운 규칙도 표현하지 못할 수 있다.

**딥러닝 코드에서는**

- 클래스 번호를 회귀 출력처럼 다루면 같은 문제 → 분류는 클래스별 출력 + CrossEntropy

### 1-3. `B` 모델을 열어 보기 · `coef_` · `intercept_` · `predict_proba` <sub>원본 셀 12 · 13</sub>

> 🎯 **레이블 방식 모델은 정말 학습을 못 한 것인가, 아니면 다른 것을 배운 것인가?**

💡 **쉽게 말하면**

- 모델은 **우리가 준 숫자의 관계를 그대로** 씀.
- blue 0 < green 1 < red 2를 진짜 순서라고 믿고 성실히 학습.

**실행 흐름**

1. 레이블 모델 다시 학습
2. 계수 · 절편 · 반복 횟수
3. 번호별 예측 확률

**먼저 예측** · blue(0) · green(1) · red(2)의 예측 확률은 어떤 순서일까? — 답은 코드 실행 결과 아래에

In [4]:
lr_label = LogisticRegression().fit(X_label, y)                           # 레이블 방식 모델
print(f"coef_      : {lr_label.coef_[0][0]:.4f}")                         # 번호 1 늘 때 점수 변화
print(f"intercept_ : {lr_label.intercept_[0]:.4f}")                       # 번호 0일 때 점수
print(f"n_iter_    : {lr_label.n_iter_[0]}  (수렴 완료)")                 # 학습은 끝까지 됨
print("입력 숫자별 예측 확률 P(target=1)")                                # 번호마다
for v, name in enumerate(LabelEncoder().fit(df_test["color"]).classes_):  # 0 blue · 1 green · 2 red
    p = lr_label.predict_proba([[v]])[0, 1]                               # 1일 확률
    print(f"  {name:6s}({v}) -> {p:.4f}")                                 # 번호 순으로 커지는지

coef_      : 0.0458
intercept_ : 0.5288
n_iter_    : 4  (수렴 완료)
입력 숫자별 예측 확률 P(target=1)
  blue  (0) -> 0.6292
  green (1) -> 0.6398
  red   (2) -> 0.6503


**결과** · 예측 정답은 blue < green < red — 번호 순서대로 커짐

- `coef_      : 0.0458` — 번호가 1 커질 때마다 점수가 조금 오름 → 번호를 크기로 씀
- `n_iter_    : 4  (수렴 완료)` — 학습을 못 한 게 아니라 끝까지 수렴함
- 세 확률이 번호 순으로 커지고 모두 0.5 이상 → 모든 입력에 1 예측
  - `blue  (0) -> 0.6292`
  - `green (1) -> 0.6398`
  - `red   (2) -> 0.6503`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code> → y</td>
<td style="text-align: left;">정답 클래스 번호 (순서 의미 없음 · 모델이 크기로 쓰지 않음)</td>
<td style="text-align: left;"><strong>원래 용도</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code> → 명목형 X</td>
<td style="text-align: left;">임의의 순서가 모델에 전달</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">모델 열어 보기</td>
<td style="text-align: left;"><code>coef_</code> · <code>predict_proba</code>로 근거 확인</td>
<td style="text-align: left;">점수가 이상할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - LabelEncoder가 나쁜 도구라는 결론으로 가지 않기 → 원래 **타깃 y용** 도구
> - 입력 명목형 X에 쓰면 임의의 순서가 전달될 수 있다는 것이 핵심

> ✅ **한 줄 정리** · 모델은 이상하게 학습한 것이 아니라, 우리가 준 번호의 순서를 진짜 순서로 믿고 그대로 쓴 것이다.

**딥러닝 코드에서는**

- `model.fc.weight` 확인 · 출력 확률 분포 확인 — 학습이 됐는지 열어 보는 습관

### 1-4. `B` 트리에서는? · `DecisionTreeClassifier` <sub>원본 셀 14</sub>

> 🎯 **같은 두 표현을 결정 트리에 넣으면 결과가 다른가?**

💡 **쉽게 말하면**

- 트리는 '번호 ≤ 0.5?' '번호 ≤ 1.5?'처럼 **기준값을 여러 번** 둘 수 있음.
- 그래서 가운데 green도 따로 떼어 냄.

**실행 흐름**

1. 같은 X_label · X_onehot
2. 트리로 학습 데이터 정확도
3. 깊이 확인

**먼저 예측** · 트리는 Label 방식에서도 1.000일까? — 답은 코드 실행 결과 아래에

In [5]:
from sklearn.tree import DecisionTreeClassifier  # 기준값으로 나누는 모델

dt_label = DecisionTreeClassifier(random_state=42).fit(X_label, y)    # 번호 열
dt_onehot = DecisionTreeClassifier(random_state=42).fit(X_onehot, y)  # 원-핫 열
print("=== 결정 트리 (트리 모델) ===")                                # 원본 출력 형식
print(f"레이블 방식 학습 데이터 정확도: {accuracy_score(y, dt_label.predict(X_label)):.3f}")  # 번호
# 원-핫
print(f"원-핫 방식 학습 데이터 정확도: {accuracy_score(y, dt_onehot.predict(X_onehot)):.3f}")
# 질문 횟수
print("레이블 트리 깊이:", dt_label.get_depth(), "· 원-핫 트리 깊이:", dt_onehot.get_depth())

=== 결정 트리 (트리 모델) ===
레이블 방식 학습 데이터 정확도: 1.000
원-핫 방식 학습 데이터 정확도: 1.000
레이블 트리 깊이: 2 · 원-핫 트리 깊이: 1


**결과** · 예측 정답은 1.000 — 기준값 두 번으로 green을 떼어 냄

- `레이블 방식 학습 데이터 정확도: 1.000` — 번호 ≤ 0.5 · 번호 ≤ 1.5 두 번으로 green만 분리
- `원-핫 방식 학습 데이터 정확도: 1.000` — green 열 하나로 바로 분리
- `레이블 트리 깊이: 2 · 원-핫 트리 깊이: 1` — 번호 방식은 질문이 한 번 더 필요

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">선형 + Label</td>
<td style="text-align: left;">직선 하나 → 가운데를 못 뗌</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">트리 + Label</td>
<td style="text-align: left;">기준값 여러 번 → 떼어 낼 수 있음</td>
<td style="text-align: left;">이번 toy 예제에서는 됨</td>
</tr>
<tr>
<td style="text-align: left;">트리 + One-Hot</td>
<td style="text-align: left;">열 하나로 바로</td>
<td style="text-align: left;">범주가 적을 때 안전한 기본</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 트리도 숫자의 순서를 기준값으로 씀 → '트리면 임의의 정수 인코딩이 항상 괜찮다'는 뜻이 아님
> - 범주가 많으면 가짜 순서 때문에 질문이 많이 필요해짐

> ✅ **한 줄 정리** · 트리는 기준값을 여러 번 둘 수 있어 이번 예제에서는 번호도 견디지만, 순서를 쓰는 것은 같으므로 명목형에는 원-핫이 기본이다.

**딥러닝 코드에서는**

- 트리 계열(XGBoost 등)은 범주를 정수로 넣기도 하지만, 신경망은 임베딩이 기본

### 1-5. `A` Ordinal 인코딩 · `OrdinalEncoder` · `categories=` <sub>원본 셀 15 ~ 18</sub>

> 🎯 **small < medium < large처럼 실제 순서가 있으면 어떻게 인코딩하는가?**

💡 **쉽게 말하면**

- 순서가 **진짜로 의미 있을 때만** 그 순서를 번호로.
- 순서는 사람이 직접 지정.

**실행 흐름**

1. 크기 열 5개
2. categories에 순서 지정
3. 변환 결과 확인

**먼저 예측** · `categories=[[small, medium, large]]`로 지정하면 large의 번호는? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.preprocessing import OrdinalEncoder  # 순서형 → 번호

df_ord = pd.DataFrame({"size": ["small", "medium", "large", "medium", "small"]})  # 순서 있는 범주
ord_enc = OrdinalEncoder(categories=[["small", "medium", "large"]])  # 순서를 직접 지정 → 0 · 1 · 2
df_ord["size_code"] = ord_enc.fit_transform(df_ord[["size"]])        # 학습 + 변환 한 번에
print(df_ord.to_string())                                            # 원본 셀 16의 표
print("지정 안 하면:", OrdinalEncoder().fit(df_ord[["size"]]).categories_[0])  # 알파벳 순이 됨

     size  size_code
0   small        0.0
1  medium        1.0
2   large        2.0
3  medium        1.0
4   small        0.0
지정 안 하면: ['large' 'medium' 'small']


**결과** · 예측 정답은 2

- `2   large        2.0` — 지정한 순서대로 large = 2
- `0   small        0.0` — small = 0
- `지정 안 하면: ['large' 'medium' 'small']` — 알파벳 순 → large가 0 · 순서가 뒤집힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">레이블 인코딩</td>
<td style="text-align: left;">번호 부여</td>
<td style="text-align: left;">(주로) <strong>타깃 y</strong> · X에 쓰면 가짜 순서 위험 (특히 선형 · 거리 모델)</td>
</tr>
<tr>
<td style="text-align: left;">원-핫 인코딩</td>
<td style="text-align: left;">red = [1, 0, 0]</td>
<td style="text-align: left;"><strong>순서 없는 명목형 X</strong> · 카테고리가 많으면 차원 폭발</td>
</tr>
<tr>
<td style="text-align: left;">오디널 인코딩</td>
<td style="text-align: left;">small 0 · medium 1 · large 2</td>
<td style="text-align: left;"><strong>순서가 의미 있을 때</strong> · 순서를 직접 지정</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Ordinal을 쓴다고 항상 성능이 좋아지는 것은 아님 → 장점은 실제 순서 표현 · 차원 유지
> - `categories=`를 빼면 알파벳 순이라 순서가 틀어짐

> ✅ **한 줄 정리** · 실제 순서가 있는 범주만 `OrdinalEncoder(categories=)`로 순서를 직접 지정해 번호를 붙인다.

**딥러닝 코드에서는**

- 학년 · 등급처럼 순서가 있는 값은 숫자 하나로, 없는 값은 임베딩으로

---
## 2. 텍스트 인코딩

> **이 장의 흐름**
>
> - 문장 6개로 단어 사전을 만들고 로지스틱 학습
> - 그다음 사전에 없는 단어로 된 새 문장 → 모델 안을 열어 근거 확인 → 사전 단어로 다시

<img src="images/pfm3_bow.png" alt="새 문장이 사전과 하나도 안 겹치면 벡터가 전부 0" width="793">

**파이프라인 위치** · ③ 전처리 — 텍스트 열

### 2-1. `B` 단어 사전 만들기 · `CountVectorizer` · `get_feature_names_out` <sub>원본 셀 19 · 20</sub>

> 🎯 **CountVectorizer는 문장을 어떤 숫자로 바꾸는가?**

💡 **쉽게 말하면**

- 문장에 단어가 **몇 번 나왔는지** 세어 벡터로.
- 단어 사전은 학습 문장에서만 만듦.

**실행 흐름**

1. 문장 6개 · 긍정 / 부정 정답
2. fit_transform으로 사전 + 벡터
3. 로지스틱 학습

**먼저 예측** · 사전에는 단어가 몇 개 들어갈까? ('이'도 들어갈까?) — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터

texts = ["이 영화 정말 재밌다", "완전 감동적이다", "너무 지루하고 별로였다",  # 문장 6개
         "시간 낭비였다", "연출이 좋았다", "스토리가 엉망이다"]
labels = [1, 1, 0, 0, 1, 0]                                    # 1 = 긍정, 0 = 부정
vectorizer = CountVectorizer()                                 # 공백으로 나눠 단어를 셈
X_text = vectorizer.fit_transform(texts)                       # 사전 만들기 + 벡터화
print("단어 사전:", list(vectorizer.get_feature_names_out()))  # 원본 셀 20
print("벡터 모양:", X_text.shape)                              # (문장 수, 사전 크기)
text_model = LogisticRegression().fit(X_text, labels)          # 감성 분류기 (toy)

단어 사전: ['감동적이다', '낭비였다', '너무', '별로였다', '스토리가', '시간', '엉망이다', '연출이', '영화', '완전', '재밌다', '정말', '좋았다', '지루하고']
벡터 모양: (6, 14)


**결과** · 예측 정답은 14개 · '이'는 한 글자라 빠짐

- `단어 사전: ['감동적이다', '낭비였다', '너무', '별로였다', '스토리가', '시간', '엉망이다', '연출이', '영화', '완전', '재밌다', '정말', '좋았다', '지루하고']` — 사전 단어 14개 · 한 글자 '이'는 기본 설정에서 단어로 안 셈
- `벡터 모양: (6, 14)` — 문장 6개 × 단어 14개

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>CountVectorizer</code></td>
<td style="text-align: left;">단어 등장 횟수</td>
<td style="text-align: left;">짧은 문장 · 기준선</td>
</tr>
<tr>
<td style="text-align: left;">형태소 분석기 · 서브워드</td>
<td style="text-align: left;">'영화는' → '영화' + '는'</td>
<td style="text-align: left;">한국어 실무</td>
</tr>
<tr>
<td style="text-align: left;">임베딩 (딥러닝)</td>
<td style="text-align: left;">단어 의미를 담은 벡터</td>
<td style="text-align: left;">의미 비교가 필요할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이 예제는 실제 한국어 감성 분석 성능이 아니라 **새 문장을 어떻게 숫자로 바꾸는지** 보는 toy 예제
> - 한국어는 공백 기준 토큰화만으로는 단어가 부정확할 수 있음

> ✅ **한 줄 정리** · CountVectorizer는 학습 문장으로 단어 사전을 만들고, 문장마다 사전 단어가 나온 횟수를 벡터로 만든다.

**딥러닝 코드에서는**

- `tokenizer(texts)` → 토큰 번호 → `nn.Embedding` — 글자를 숫자로 바꾸는 같은 단계

### 2-2. `A` 새 문장 · 출력 ≠ 근거 · `transform` · `nnz` · `predict_proba` <sub>원본 셀 21 ~ 25</sub>

> 🎯 **원본에서 '이 영화는 별로다'를 부정(0)으로 맞힌 것은 정말 맞힌 것인가?**

💡 **쉽게 말하면**

- 새 문장은 **transform만** (fit하면 사전이 바뀜).
- 그런데 사전과 겹치는 단어가 0개면 벡터는 전부 0.

**실행 흐름**

1. 새 문장 transform
2. 예측
3. 겹친 단어 수 · 확률 열어 보기

**먼저 예측** · 모델이 부정(0)으로 맞힐까? 그 근거는? — 답은 코드 실행 결과 아래에

In [8]:
new_X = vectorizer.transform(["이 영화는 별로다"])  # 사전은 그대로 · transform만
feature_names = vectorizer.get_feature_names_out()                              # 사전 단어
print("예측 결과:", text_model.predict(new_X))                                  # 원본 셀 22
print("사전에서 매칭된 단어 수:", new_X.nnz, "개")                              # 0이 아닌 칸 수
print("매칭된 단어           :", list(feature_names[new_X.toarray()[0] > 0]))   # 겹친 단어
print("예측 확률 [부정, 긍정] :", text_model.predict_proba(new_X)[0].round(4))  # 근거가 있나

예측 결과: [1]
사전에서 매칭된 단어 수: 0 개
매칭된 단어           : []
예측 확률 [부정, 긍정] : [0.5 0.5]


**결과** · 예측 정답은 근거 없음 — 확률 [0.5, 0.5] · 이 환경에서는 출력마저 1(긍정)로 바뀜

- `예측 결과: [1]` — 원본(sklearn 1.6)은 [0] · 이 환경(1.9)은 [1] → 동점이라 버전에 따라 뒤집힘
- `사전에서 매칭된 단어 수: 0 개` — '영화는' ≠ '영화', '별로다' ≠ '별로였다', '이'는 단어로 안 셈
- `예측 확률 [부정, 긍정] : [0.5 0.5]` — 완전히 반반 → 아무 판단도 못 한 상태

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>fit_transform</code></td>
<td style="text-align: left;">사전을 새로 만들고 변환</td>
<td style="text-align: left;">학습 문장에만</td>
</tr>
<tr>
<td style="text-align: left;"><code>transform</code></td>
<td style="text-align: left;">기존 사전으로 변환</td>
<td style="text-align: left;">새 문장 · test</td>
</tr>
<tr>
<td style="text-align: left;">확률 · 겹친 단어 확인</td>
<td style="text-align: left;">출력의 근거를 봄</td>
<td style="text-align: left;">예측이 그럴듯해도 항상</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - **출력이 나왔다는 것과 그 출력에 근거가 있다는 것은 다름**
> - 이 감각은 3-2회차에서 다시 만남 · '정확도 90%' → 필요한 일을 잘하고 있나?

> ✅ **한 줄 정리** · 사전에 없는 단어뿐인 문장은 벡터가 0이 되어 확률 [0.5, 0.5]이므로, 그 출력은 맞힌 것이 아니라 근거 없는 동점이다.

**딥러닝 코드에서는**

- 토크나이저에 없는 단어는 `[UNK]` → 너무 많으면 모델이 문장을 못 읽음

### 2-3. `B` 사전에 있는 단어로 · `transform` · `predict_proba` <sub>원본 셀 26</sub>

> 🎯 **사전에 있는 단어로 쓰면 모델의 판단에 근거가 생기는가?**

💡 **쉽게 말하면**

- 사전 단어가 겹치면 그 단어의 가중치가 **근거**가 됨.
- 확률이 반반에서 움직임.

**실행 흐름**

1. 사전 단어로 된 문장
2. 겹친 단어 · 확률 · 예측

**먼저 예측** · '정말 지루하고 별로였다'의 부정 확률은 0.5보다 클까? — 답은 코드 실행 결과 아래에

In [9]:
better_X = vectorizer.transform(["정말 지루하고 별로였다"])                        # 사전 단어 3개
print("사전에서 매칭된 단어 수:", better_X.nnz, "개")                              # 겹친 단어 수
print("매칭된 단어           :", list(feature_names[better_X.toarray()[0] > 0]))   # 어떤 단어
print("예측 확률 [부정, 긍정] :", text_model.predict_proba(better_X)[0].round(4))  # 근거가 생김
print("예측 결과             :", text_model.predict(better_X)[0])                  # 부정

사전에서 매칭된 단어 수: 3 개
매칭된 단어           : ['별로였다', '정말', '지루하고']
예측 확률 [부정, 긍정] : [0.5728 0.4272]
예측 결과             : 0


**결과** · 예측 정답은 큼 — 0.5728

- `사전에서 매칭된 단어 수: 3 개` — 세 단어 모두 사전에 있음
- `매칭된 단어           : ['별로였다', '정말', '지루하고']` — '정말'은 긍정 문장에서 나온 단어
- `예측 확률 [부정, 긍정] : [0.5728 0.4272]` — 부정 쪽으로 기울었지만 문장 6개로 배운 만큼만

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">겹친 단어 0개</td>
<td style="text-align: left;">확률 [0.5, 0.5] · 근거 없음</td>
<td style="text-align: left;">결과를 믿지 않기</td>
</tr>
<tr>
<td style="text-align: left;">겹친 단어 3개</td>
<td style="text-align: left;">가중치 합이 근거</td>
<td style="text-align: left;">판단이 생김 (그래도 약함)</td>
</tr>
<tr>
<td style="text-align: left;">데이터 늘리기</td>
<td style="text-align: left;">사전 · 가중치가 풍부해짐</td>
<td style="text-align: left;">실제 서비스</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - '정말'은 긍정 문장에만 나와 긍정 쪽 가중치 → 부정 문장에서 확률을 끌어내림

> ✅ **한 줄 정리** · 사전 단어가 겹쳐야 가중치가 근거가 되어 확률이 반반에서 움직인다.

**딥러닝 코드에서는**

- 어텐션 · 기여도 시각화로 어떤 토큰이 판단 근거였는지 확인

---
## 3. 스케일링

> **이 장의 흐름**
>
> - 2회차 펭귄 KNN 0.7801의 이유(단위 차이)를 다시 봄 → 키 · 용돈으로 StandardScaler
> - 이상치에는 RobustScaler, Iris KNN에서 MinMax vs Standard 비교

<img src="images/pfm3_scale.png" alt="스케일링 전후 · 큰 단위 열이 거리를 지배하지 않게" width="911">

**파이프라인 위치** · ③ 전처리 — 숫자 열

### 3-1. `A` 지난 숙제 · 펭귄 단위 차이 · `describe()` · `거리` <sub>원본 셀 27</sub>

> 🎯 **2회차에서 KNN만 0.7801로 낮았던 이유를 숫자로 확인하면?**

💡 **쉽게 말하면**

- 부리 길이는 32 ~ 60, 체중은 2700 ~ 6300.
- 거리를 재면 **체중 차이만** 보임.

**실행 흐름**

1. 펭귄 측정값 범위
2. 두 마리 사이 거리에서 열마다 몫

**먼저 예측** · 두 펭귄 거리에서 체중이 차지하는 몫은 몇 %일까? — 답은 코드 실행 결과 아래에

In [10]:
CSV = "../02_모델선택_펭귄/penguins.csv"                            # 원본 폴더 데이터
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]  # 측정값 4개
peng = pd.read_csv(CSV).dropna(subset=meas).reset_index(drop=True)  # 2회차와 같은 342마리
rng_tbl = peng[meas].agg(["min", "max"]).T                          # 열마다 최소 · 최대
rng_tbl["범위"] = rng_tbl["max"] - rng_tbl["min"]                   # 크기 차이
print(rng_tbl.round(1).to_string())                                 # 범위 비교
d = (peng.loc[0, meas] - peng.loc[1, meas]).astype(float)           # 첫 두 마리의 차이
share = (d ** 2) / (d ** 2).sum()                                   # 거리²에서 열마다 몫
print("거리 몫:", share.round(4).to_dict())                         # 체중이 거의 전부

                      min     max      범위
bill_length_mm       32.1    59.6    27.5
bill_depth_mm        13.1    21.5     8.4
flipper_length_mm   172.0   231.0    59.0
body_mass_g        2700.0  6300.0  3600.0
거리 몫: {'bill_length_mm': 0.0001, 'bill_depth_mm': 0.0007, 'flipper_length_mm': 0.0099, 'body_mass_g': 0.9894}


**결과** · 예측 정답은 98.94%

- `bill_length_mm      32.1    59.6    27.5` — 부리 길이 범위 약 28
- `body_mass_g       2700.0  6300.0  3600.0` — 체중 범위 3600 → 부리 범위의 백 배가 넘음
- `'body_mass_g': 0.9894` — 두 마리 거리의 98.94%가 체중 차이 → KNN은 사실상 체중만 봄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">거리 기반 (KNN · SVM)</td>
<td style="text-align: left;">큰 단위 열이 거리를 지배</td>
<td style="text-align: left;">스케일링 필수</td>
</tr>
<tr>
<td style="text-align: left;">선형 · 경사하강 (로지스틱 · 신경망)</td>
<td style="text-align: left;">학습 속도 · 규제에 영향</td>
<td style="text-align: left;">스케일링 권장</td>
</tr>
<tr>
<td style="text-align: left;">트리 계열</td>
<td style="text-align: left;">기준값만 씀 · 단위 무관</td>
<td style="text-align: left;">스케일링 거의 불필요</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 27의 2회차 표는 로지스틱 CV를 0.9927 ± 0.0089로 적었지만, 같은 설정의 파일별 2 실행값은 0.9891 ± 0.0145
> - KNN 0.7801 ± 0.0355는 같음

> ✅ **한 줄 정리** · 단위가 큰 체중 열이 거리의 98.94%를 차지해, 스케일링 전 KNN은 사실상 체중만 보고 이웃을 찾았다.

**딥러닝 코드에서는**

- 이미지 픽셀 0 ~ 255 → 0 ~ 1 · 평균 · 표준편차 정규화 — 같은 이유

### 3-2. `A` StandardScaler · `fit_transform` · `mean_` · `scale_` <sub>원본 셀 28 ~ 30</sub>

> 🎯 **키(150 ~ 170)와 용돈(1,000 ~ 50,000)을 StandardScaler로 바꾸면 어떻게 되는가?**

💡 **쉽게 말하면**

- 열마다 **평균을 빼고 표준편차로 나눔** → 평균 0 · 표준편차 1.
- 수학 0 ~ 100점, 음악 0 ~ 1억 점이면 음악만 중요해지는 상황을 막음.

**실행 흐름**

1. 키 · 용돈 3명
2. StandardScaler fit_transform
3. 전후 표 · 평균 · 표준편차

**먼저 예측** · 스케일링 후 값은 -1 ~ 1 안에만 있을까? — 답은 코드 실행 결과 아래에

In [11]:
from sklearn.preprocessing import StandardScaler  # 평균 0 · 표준편차 1

X_money = np.array([[150, 1000], [160, 20000], [170, 50000]])  # 키(cm) · 용돈(원)
scaler = StandardScaler()                                      # 열마다 따로 계산
X_scaled = scaler.fit_transform(X_money)                       # 평균 · 표준편차를 배우고 변환
df_scale = pd.DataFrame(np.hstack([X_money, X_scaled]),        # 전후를 옆으로
                        columns=["키(cm)", "용돈(원)", "키(스케일링 후)", "용돈(스케일링 후)"])
print(df_scale.round(6).to_string())                           # 원본 셀 29의 표
# 배운 값
print("열 평균:", scaler.mean_.round(1).tolist(), "· 열 표준편차:", scaler.scale_.round(1).tolist())

   키(cm)    용돈(원)  키(스케일링 후)  용돈(스케일링 후)
0  150.0   1000.0  -1.224745   -1.123698
1  160.0  20000.0   0.000000   -0.181775
2  170.0  50000.0   1.224745    1.305473
열 평균: [160.0, 23666.7] · 열 표준편차: [8.2, 20171.5]


**결과** · 예측 정답은 아님 — 1.305처럼 1을 넘을 수 있음

- `0  150.0   1000.0  -1.224745   -1.123698` — 키 · 용돈이 비슷한 크기로
- `2  170.0  50000.0   1.224745    1.305473` — 1.305 → 범위가 -1 ~ 1로 묶이는 것이 아님
- `열 평균: [160.0, 23666.7] · 열 표준편차: [8.2, 20171.5]` — fit이 배운 기준 (test에도 이 값으로)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">평균 0 · 표준편차 1 (z-score)</td>
<td style="text-align: left;"><strong>기본 선택</strong> (선형 · 거리 기반)</td>
</tr>
<tr>
<td style="text-align: left;">MinMaxScaler (3-4)</td>
<td style="text-align: left;">[0, 1] 구간 · 이상치에 민감</td>
<td style="text-align: left;">범위가 중요한 값 (픽셀 등)</td>
</tr>
<tr>
<td style="text-align: left;">RobustScaler (3-3)</td>
<td style="text-align: left;">중앙값 · IQR 기반 · 이상치에 강함</td>
<td style="text-align: left;">이상치가 있을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 29 주석 '평균 0, 표준편차 1 (-1 ~ 1)'은 틀림 → 값은 -1 ~ 1을 넘을 수 있음 (1.305)
> - fit은 기준(평균 · 표준편차)을 배우는 것 → train에만 (5장)

> ✅ **한 줄 정리** · StandardScaler는 열마다 평균을 빼고 표준편차로 나눠 크기를 맞추며, 결과 범위가 -1 ~ 1로 정해지지는 않는다.

**딥러닝 코드에서는**

- `transforms.Normalize(mean, std)` — 채널마다 같은 계산
- `nn.BatchNorm` — 층 안에서 같은 생각

### 3-3. `B` RobustScaler와 이상치 · `RobustScaler` · `중앙값 · IQR` <sub>원본 셀 31 · 32</sub>

> 🎯 **이상치 하나가 섞이면 StandardScaler와 RobustScaler는 어떻게 다르게 바꾸는가?**

💡 **쉽게 말하면**

- 평균은 이상치에 **끌려감** → 정상 값 세 개가 한쪽에 뭉침.
- 중앙값 · IQR은 이상치에 덜 흔들림 → 정상 값 사이 간격이 살아 있음.

**실행 흐름**

1. 정상 3개 + 이상치 1개
2. Standard · Robust 변환
3. 전후 표 비교

**먼저 예측** · 정상 값 1 · 2 · 3은 StandardScaler 후에도 간격이 잘 보일까? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.preprocessing import RobustScaler  # 중앙값 · IQR 기반

X_out = np.array([[1., 10.], [2., 12.], [3., 13.], [100., 1000.]])  # 마지막 행이 이상치
sc = StandardScaler().fit_transform(X_out)                 # 평균 · 표준편차 → 이상치에 민감
rb = RobustScaler().fit_transform(X_out)                   # 중앙값 · IQR → 덜 민감
df_rb = pd.DataFrame(np.hstack([X_out, sc, rb]),           # 전후를 옆으로
                     columns=["x1", "x2", "Std_x1", "Std_x2", "Robust_x1", "Robust_x2"])
print(df_rb.round(6).to_string())                          # 원본 셀 32의 표
gap_sc, gap_rb = sc[2, 0] - sc[0, 0], rb[2, 0] - rb[0, 0]  # 정상 값 1과 3 사이 간격
print("정상 3개 폭 · Std:", round(gap_sc, 3), "· Robust:", round(gap_rb, 3))  # 간격 비교

      x1      x2    Std_x1    Std_x2  Robust_x1  Robust_x2
0    1.0    10.0 -0.600832 -0.581243  -0.058824  -0.010070
1    2.0    12.0 -0.577270 -0.576570  -0.019608  -0.002014
2    3.0    13.0 -0.553708 -0.574233   0.019608   0.002014
3  100.0  1000.0  1.731810  1.732045   3.823529   3.977845
정상 3개 폭 · Std: 0.047 · Robust: 0.078


**결과** · 예측 정답은 잘 안 보임 — -0.60 ~ -0.55에 뭉침

- `0    1.0    10.0 -0.600832 -0.581243  -0.058824  -0.010070` — 정상 값이 Std에서는 -0.6 근처에 뭉침
- `3  100.0  1000.0  1.731810  1.732045   3.823529   3.977845` — Robust에서는 이상치가 3.8로 멀리 떨어져 보임
- `정상 3개 폭 · Std: 0.047 · Robust: 0.078` — Robust가 정상 값 사이 간격을 더 남김

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">평균 · 표준편차 → 이상치가 기준을 끌고 감</td>
<td style="text-align: left;">이상치가 거의 없을 때</td>
</tr>
<tr>
<td style="text-align: left;">RobustScaler</td>
<td style="text-align: left;">중앙값 · IQR → 정상 값 기준</td>
<td style="text-align: left;">이상치가 있을 때</td>
</tr>
<tr>
<td style="text-align: left;">이상치 제거 · 클리핑</td>
<td style="text-align: left;">값 자체를 고침</td>
<td style="text-align: left;">측정 오류로 확인됐을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Robust도 이상치를 없애 주지는 않음 → 이상치는 여전히 큰 값으로 남음 (3.8)

> ✅ **한 줄 정리** · 이상치가 있으면 평균 · 표준편차가 끌려가므로, 중앙값 · IQR을 쓰는 RobustScaler가 정상 값의 간격을 더 잘 지킨다.

**딥러닝 코드에서는**

- 손실이 튀는 이상치에는 `HuberLoss` · gradient clipping — 같은 '덜 흔들리게' 생각

### 3-4. `A` MinMax vs Standard · Iris KNN · `MinMaxScaler` · `Pipeline` · `f1_score` <sub>원본 셀 33 ~ 37</sub>

> 🎯 **Iris KNN에서 MinMax와 Standard 중 어느 쪽이 좋다고 결론 내릴 수 있는가?**

💡 **쉽게 말하면**

- MinMax는 [0, 1]로, Standard는 평균 0 · 표준편차 1로.
- 한 번의 분할 결과로 '이 데이터엔 이게 좋다'고 정하지 않음.

**실행 흐름**

1. Iris 7 : 3 분할
2. 스케일러만 다른 Pipeline 두 개
3. 정확도 · F1(macro) 비교

**먼저 예측** · 두 스케일러의 정확도는 같을까? — 답은 코드 실행 결과 아래에

In [13]:
from sklearn.datasets import load_iris                # Iris
from sklearn.metrics import f1_score                  # 클래스별 F1의 평균(macro)
from sklearn.model_selection import train_test_split  # 분할
from sklearn.neighbors import KNeighborsClassifier    # 거리 기반 모델
from sklearn.pipeline import Pipeline                 # 전처리 + 모델 묶기
from sklearn.preprocessing import MinMaxScaler        # [0, 1]

X_iris, y_iris = load_iris(return_X_y=True)                        # 150 × 4
X_train, X_test, y_train, y_test = train_test_split(               # 원본과 같은 7 : 3
    X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=42)
# MinMax
pipe_minmax = Pipeline([("scaler", MinMaxScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
# Standard
pipe_std = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
for tag, pipe in [("MinMax", pipe_minmax), ("Std   ", pipe_std)]:  # 같은 코드로 둘 다
    pred = pipe.fit(X_train, y_train).predict(X_test)              # train에 fit · test 예측
    acc, f1 = accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")  # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")            # 원본 형식

[MinMax] Acc=0.933, F1(macro)=0.933
[Std   ] Acc=0.911, F1(macro)=0.910


**결과** · 예측 정답은 다름 — MinMax 0.933 · Standard 0.911

- `[MinMax] Acc=0.933, F1(macro)=0.933` — test 45개 중 42개
- `[Std   ] Acc=0.911, F1(macro)=0.910` — 45개 중 41개 · 1개 차이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">MinMaxScaler</td>
<td style="text-align: left;">[0, 1] · 최솟값 · 최댓값 기반 · 이상치에 민감</td>
<td style="text-align: left;">범위가 중요한 값 · 데이터가 깔끔할 때</td>
</tr>
<tr>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">평균 0 · 표준편차 1</td>
<td style="text-align: left;">일반적인 비교를 시작할 때</td>
</tr>
<tr>
<td style="text-align: left;">RobustScaler</td>
<td style="text-align: left;">중앙값 · IQR</td>
<td style="text-align: left;">이상치의 영향을 줄이고 싶을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 한 번의 split(1개 차이)으로 'Iris에는 MinMax가 더 좋다'고 결론 내리지 않음 · 인과도 증명 안 됨
> - 세 개 중 하나를 고르고 끝내는 게 아니라 **후보를 정해 비교**하는 것이 출발점 (종합 실습)

> ✅ **한 줄 정리** · 스케일러는 단위 차이가 거리를 지배하는 문제를 줄여 주며, 어느 것이 나은지는 데이터와 모델에 따라 비교해서 정한다.

**딥러닝 코드에서는**

- 입력 정규화 방식도 실험으로 비교 (0 ~ 1 vs 평균 · 표준편차)

---
## 4. 정규화

> **이 장의 흐름**
>
> - 행(샘플)마다 길이를 1로 맞춤 → 정규화한 벡터의 내적 = 코사인 유사도
> - 그다음 CountVectorizer + 코사인으로 문장 유사도

<img src="images/pfm3_norm.png" alt="정규화 = 길이는 지우고 방향만 남기기" width="434">

**파이프라인 위치** · ③ 전처리 — 텍스트 · 추천 · 임베딩

### 4-1. `B` Normalizer · 행 길이를 1로 · `Normalizer` · `L2 norm` <sub>원본 셀 38 · 39</sub>

> 🎯 **정규화는 무엇을 1로 맞추고, 스케일링과 기준이 어떻게 다른가?**

💡 **쉽게 말하면**

- **길이**는 '얼마나 많이', **방향**은 '어떤 비율로'.
- 록 100곡 + 재즈 50곡과 록 10곡 + 재즈 5곡은 길이는 달라도 취향(방향)은 같음.

**실행 흐름**

1. 벡터 3개
2. Normalizer로 행마다 길이 1
3. 전후 표 · 길이 확인

**먼저 예측** · [3, 4]는 정규화 후 어떤 값이 될까? — 답은 코드 실행 결과 아래에

In [14]:
from sklearn.preprocessing import Normalizer  # 행(샘플)마다 길이 1

X_vec = np.array([[3, 4], [1, 2], [10, 0]])                         # 샘플 3개
X_norm = Normalizer().fit_transform(X_vec)                          # 행마다 길이로 나눔 (L2)
cols = ["x1", "x2", "x1(정규화 후)", "x2(정규화 후)"]               # 원본 열 이름
df_norm = pd.DataFrame(np.hstack([X_vec, X_norm]), columns=cols)    # 전후를 옆으로
print(df_norm.round(6).to_string())                                 # 원본 셀 39의 표
print("행 길이:", np.linalg.norm(X_norm, axis=1).round(3))          # 전부 1
music = Normalizer().fit_transform(np.array([[100, 50], [10, 5]]))  # 록 · 재즈 곡 수
print("취향 A · B 정규화:", music.round(4).tolist())                # 같은 방향

     x1   x2  x1(정규화 후)  x2(정규화 후)
0   3.0  4.0   0.600000   0.800000
1   1.0  2.0   0.447214   0.894427
2  10.0  0.0   1.000000   0.000000
행 길이: [1. 1. 1.]
취향 A · B 정규화: [[0.8944, 0.4472], [0.8944, 0.4472]]


**결과** · 예측 정답은 [0.6, 0.8] — 길이 5로 나눔

- `0   3.0  4.0       0.600000       0.800000` — 3 ÷ 5 = 0.6 · 4 ÷ 5 = 0.8
- `행 길이: [1. 1. 1.]` — 모든 행의 길이가 1
- `취향 A · B 정규화: [[0.8944, 0.4472], [0.8944, 0.4472]]` — 곡 수(길이)는 10배 달라도 같은 취향으로 인식

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">스케일링</td>
<td style="text-align: left;"><strong>열(특성)</strong> 기준 · 단위 · 범위</td>
<td style="text-align: left;">스케일에 민감한 모델 (거리 · 선형)</td>
</tr>
<tr>
<td style="text-align: left;">정규화</td>
<td style="text-align: left;"><strong>행(샘플)</strong> 기준 · 벡터 길이(L2 norm)</td>
<td style="text-align: left;">방향(패턴) 비교 · 텍스트 · 추천 · 임베딩</td>
</tr>
<tr>
<td style="text-align: left;">트리 계열</td>
<td style="text-align: left;">스케일 차이 영향 거의 없음</td>
<td style="text-align: left;">둘 다 보통 불필요</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - '정규화'는 분야마다 뜻이 섞여 쓰임 (MinMax를 정규화라 부르기도) → 여기서는 행 길이를 1로

> ✅ **한 줄 정리** · 정규화는 행(샘플)마다 길이를 1로 맞춰 길이 영향은 지우고 방향(패턴)만 남긴다.

**딥러닝 코드에서는**

- `F.normalize(emb, dim=1)` — 임베딩 길이를 1로 맞춘 뒤 비교

### 4-2. `B` 정규화 내적 = 코사인 유사도 · `np.dot` · `cosine_similarity` <sub>원본 셀 40 ~ 42</sub>

> 🎯 **정규화한 두 벡터의 내적은 왜 코사인 유사도와 같은가?**

💡 **쉽게 말하면**

- 코사인 = 내적 ÷ (길이 × 길이).
- 길이를 1로 맞추면 분모가 1 → **내적 = 코사인**.

**실행 흐름**

1. 평행하지 않고 길이도 다른 A · B
2. 원래 내적 · 정규화 내적 · cosine_similarity

**먼저 예측** · 세 값 중 같은 것은? — 답은 코드 실행 결과 아래에

In [15]:
from sklearn.metrics.pairwise import cosine_similarity  # 방향 유사도

A = np.array([[3., 4.]])                                                 # 길이 5
B = np.array([[8., 6.]])  # 길이 10 · 평행하지 않음 (평행하면 코사인이 항상 1)
normer = Normalizer()                                                    # 길이 1로
A_n, B_n = normer.fit_transform(A), normer.transform(B)                  # 행마다 따로 나눔
print(f"원래 벡터의 내적        : {float(np.dot(A[0], B[0])):.4f}")      # 길이 · 방향 둘 다 반영
print(f"정규화된 벡터의 내적    : {float(np.dot(A_n[0], B_n[0])):.4f}")  # 방향만
print(f"cosine_similarity      : {float(cosine_similarity(A, B)[0, 0]):.4f}")  # 방향만

원래 벡터의 내적        : 48.0000
정규화된 벡터의 내적    : 0.9600
cosine_similarity      : 0.9600


**결과** · 예측 정답은 뒤의 두 값 (0.9600)

- `원래 벡터의 내적        : 48.0000` — 3 × 8 + 4 × 6 = 48 · 길이가 섞임
- `정규화된 벡터의 내적    : 0.9600` — 48 ÷ (5 × 10) = 0.96
- `cosine_similarity      : 0.9600` — 정규화 내적과 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내적</td>
<td style="text-align: left;">길이 × 길이 × 코사인</td>
<td style="text-align: left;">길이도 의미가 있을 때</td>
</tr>
<tr>
<td style="text-align: left;">코사인 유사도</td>
<td style="text-align: left;">방향만 · 같으면 1 · 직각 0 · 반대 -1</td>
<td style="text-align: left;">텍스트 · 임베딩 비교</td>
</tr>
<tr>
<td style="text-align: left;">정규화 + 내적</td>
<td style="text-align: left;">코사인과 같고 계산이 빠름</td>
<td style="text-align: left;">검색 · 추천에서 대량 비교</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 평행한 벡터로 실험하면 코사인이 항상 1이라 차이가 안 보임 → 원본은 [8, 6]을 골랐음

> ✅ **한 줄 정리** · 길이를 1로 맞추면 분모가 사라져, 정규화한 벡터의 내적이 곧 코사인 유사도다.

**딥러닝 코드에서는**

- CLIP · 문장 임베딩 검색 = 정규화 + 내적(행렬곱 한 번)

### 4-3. `B` 문장 유사도 · `CountVectorizer` · `cosine_similarity` <sub>원본 셀 43 ~ 45</sub>

> 🎯 **'영화가 좋다'와 '영화가 진짜로 좋다'의 코사인 유사도는? 그 뜻은?**

💡 **쉽게 말하면**

- 문장 → 단어 빈도 벡터 → **방향**이 얼마나 비슷한가.
- 단어 사용 패턴이 비슷하다는 뜻이지, 의미가 같다는 뜻은 아님.

**실행 흐름**

1. 두 문장 벡터화
2. 코사인 유사도 (안에서 정규화)
3. 단어를 바꾼 문장과 비교

**먼저 예측** · 단어 2개가 겹치고 1개가 다르면 유사도는? — 답은 코드 실행 결과 아래에

In [16]:
pair = CountVectorizer().fit(["영화가 좋다", "영화가 진짜로 좋다"])            # 두 문장 사전
V = pair.transform(["영화가 좋다", "영화가 진짜로 좋다"])                      # 단어 빈도 벡터
print("단어 사전:", list(pair.get_feature_names_out()))                        # 단어 3개
print("코사인 유사도:", round(float(cosine_similarity(V[0], V[1])[0, 0]), 4))  # 안에서 정규화함
W = pair.transform(["영화가 좋다", "영화가 별로다"])  # 반대 뜻 문장 ('별로다'는 사전에 없음)
print("반대 뜻 문장 유사도:", round(float(cosine_similarity(W[0], W[1])[0, 0]), 4))  # 의미와 무관

단어 사전: ['영화가', '좋다', '진짜로']
코사인 유사도: 0.8165
반대 뜻 문장 유사도: 0.7071


**결과** · 예측 정답은 0.8165 = 2 ÷ (√2 × √3)

- `단어 사전: ['영화가', '좋다', '진짜로']` — 단어 3개
- `코사인 유사도: 0.8165` — 겹친 단어 2 ÷ (√2 × √3) → 단어 패턴이 비슷
- `반대 뜻 문장 유사도: 0.7071` — 뜻은 반대인데 0.71 → 단어가 겹친다는 뜻일 뿐 의미가 같다는 뜻이 아님

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Bag-of-Words + 코사인</td>
<td style="text-align: left;">단어 사용 패턴의 방향</td>
<td style="text-align: left;">단어가 겹치는지 볼 때</td>
</tr>
<tr>
<td style="text-align: left;">LLM 임베딩 + 코사인</td>
<td style="text-align: left;">의미를 담은 벡터의 방향</td>
<td style="text-align: left;">의미 비교 · 검색</td>
</tr>
<tr>
<td style="text-align: left;"><code>cosine_similarity</code></td>
<td style="text-align: left;">안에서 정규화 → Normalizer 따로 필요 없음</td>
<td style="text-align: left;">두 벡터 묶음 비교</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 코사인 유사도가 높다 = 이 공간에서 방향이 비슷하다 · '의미가 같다'와 다름

> ✅ **한 줄 정리** · CountVectorizer + 코사인은 단어 사용 패턴의 방향을 비교하며, LLM 임베딩도 같은 아이디어를 더 풍부한 벡터로 쓴다.

**딥러닝 코드에서는**

- 문장 임베딩 모델 → `F.normalize` → 내적으로 비슷한 문서 찾기 (RAG 검색)

---
## 5. 데이터 누수

> **이 장의 흐름**
>
> - Iris에서 전체로 fit한 scaler vs Pipeline (한 번 분할 · 교차검증)
> - 그다음 난수 데이터에서 '점수가 안 올라도 문제'에서 '없는 패턴을 87%로 보이게 함'까지

<img src="images/pfm3_leak.png" alt="fit 범위 · 누수는 전체로, Pipeline은 fold의 학습 부분으로만" width="769">

**파이프라인 위치** · ③ 전처리 → ⑤ 교차검증 — fit 범위

### 5-1. `A` 전체로 fit한 scaler vs Pipeline · `fit(X)` · `Pipeline` <sub>원본 셀 46 ~ 52</sub>

> 🎯 **test까지 포함해 scaler를 맞추면 점수가 더 높아지는가?**

💡 **쉽게 말하면**

- 시험지를 **미리 훔쳐보고** 공부 기준을 정한 것.
- 점수가 안 올라도 그 시험은 무효.

**실행 흐름**

1. 전체 X로 scaler fit (나쁜 예)
2. train · test를 그 scaler로 변환 → KNN
3. Pipeline(train에만 fit) → KNN

**먼저 예측** · 누수 방식의 점수가 더 높을까? — 답은 코드 실행 결과 아래에

In [17]:
scaler_bad = StandardScaler().fit(X_iris)                # (나쁜 예) test까지 포함한 평균 · 표준편차
X_train_bad = scaler_bad.transform(X_train)              # 누수된 기준으로 변환
X_test_bad = scaler_bad.transform(X_test)                # 같은 기준
knn_bad = KNeighborsClassifier(n_neighbors=5).fit(X_train_bad, y_train)  # 누수 데이터로 KNN
pred_bad = knn_bad.predict(X_test_bad)                   # 누수 기준의 test 예측
pipe = Pipeline([("scaler", StandardScaler()),           # (올바른 예) train에만 fit
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
pred_pipe = pipe.fit(X_train, y_train).predict(X_test)   # scaler도 train에서만 fit
for tag, p in [("누수 발생·KNN", pred_bad), ("Pipeline·KNN ", pred_pipe)]:  # 같은 형식으로
    acc, f1 = accuracy_score(y_test, p), f1_score(y_test, p, average="macro")  # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")  # 원본 형식
mean_gap = scaler_bad.mean_ - pipe.named_steps["scaler"].mean_  # 두 scaler가 배운 평균의 차이
print("평균 차이(누수 - train):", mean_gap.round(3))     # test 정보만큼 기준이 다름

[누수 발생·KNN] Acc=0.911, F1(macro)=0.910
[Pipeline·KNN ] Acc=0.911, F1(macro)=0.910
평균 차이(누수 - train): [-0.03   0.002 -0.027 -0.006]


**결과** · 예측 정답은 아님 — 둘 다 0.911

- `[누수 발생·KNN] Acc=0.911, F1(macro)=0.910` — test 정보가 기준에 섞였지만 점수는 같음
- `[Pipeline·KNN ] Acc=0.911, F1(macro)=0.910` — train에서만 배운 기준
- `평균 차이(누수 - train):` — 점수는 같아도 scaler가 배운 기준(평균)은 다름 = test 정보가 들어감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>fit</code></td>
<td style="text-align: left;"><strong>훈련 데이터에만</strong></td>
<td style="text-align: left;">시험지 정보가 기준에 섞이면 안 됨 (누수)</td>
</tr>
<tr>
<td style="text-align: left;"><code>transform</code></td>
<td style="text-align: left;"><strong>훈련 · 시험 둘 다</strong></td>
<td style="text-align: left;">같은 자로 재야 함 (일관성)</td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline</code></td>
<td style="text-align: left;">fit 범위를 자동으로 train에 묶음</td>
<td style="text-align: left;">전처리가 있으면 항상</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 'test에 fit하지 마라'를 'test는 건드리지 마라'로 넓히면 안 됨 → `transform`을 빼먹으면 조용히 망가짐
> - Pipeline 밖에서 미리 fit한 처리(PCA 등)나 시험 결과를 보고 내린 선택은 Pipeline이 못 막음

> ✅ **한 줄 정리** · 전처리의 fit은 train에만, transform은 train · test 둘 다 하며, Pipeline이 이 fit 범위를 지켜 준다.

**딥러닝 코드에서는**

- 정규화 평균 · 표준편차는 train 이미지에서만 계산해 val · test에 그대로 적용

### 5-2. `B` 로지스틱에서도 · 한 번 분할 · `LogisticRegression` · `Pipeline` <sub>원본 셀 53 ~ 55</sub>

> 🎯 **스케일링 영향을 더 받는 로지스틱에서는 누수 vs Pipeline 결과가 다른가?**

💡 **쉽게 말하면**

- 로지스틱은 크기에 민감한 편 → 같은 비교를 한 번 더.
- 점수가 같아도 절차가 틀리면 무효.

**실행 흐름**

1. 누수 scaler로 변환한 데이터로 로지스틱
2. Pipeline 로지스틱
3. 비교

**먼저 예측** · 로지스틱에서는 누수 쪽이 더 높을까? — 답은 코드 실행 결과 아래에

In [18]:
lr_bad = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_bad, y_train)  # 누수 데이터
pred_lr_bad = lr_bad.predict(X_test_bad)                                        # 누수 기준의 test
pipe_lr = Pipeline([("scaler", StandardScaler()),                               # train에만 fit
                    ("lr", LogisticRegression(max_iter=5000, random_state=42))])
pred_lr_pipe = pipe_lr.fit(X_train, y_train).predict(X_test)                    # 올바른 절차
for tag, p in [("누수 발생·LR", pred_lr_bad), ("Pipeline·LR ", pred_lr_pipe)]:  # 같은 형식
    acc, f1 = accuracy_score(y_test, p), f1_score(y_test, p, average="macro")   # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")                         # 원본 형식

[누수 발생·LR] Acc=0.911, F1(macro)=0.911
[Pipeline·LR ] Acc=0.911, F1(macro)=0.911


**결과** · 예측 정답은 같음 — 둘 다 0.911

- `[누수 발생·LR] Acc=0.911, F1(macro)=0.911` — 누수 방식
- `[Pipeline·LR ] Acc=0.911, F1(macro)=0.911` — 올바른 방식 · 이번엔 같은 점수

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Iris처럼 단순한 데이터</td>
<td style="text-align: left;">차이가 작아 보일 수 있음</td>
<td style="text-align: left;">그래도 누수 방식은 원칙적으로 금지</td>
</tr>
<tr>
<td style="text-align: left;">점수 차이로 판단</td>
<td style="text-align: left;">같으면 괜찮다고 착각</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">절차로 판단</td>
<td style="text-align: left;">평가용 데이터 정보를 미리 썼는가</td>
<td style="text-align: left;"><strong>누수 판단 기준</strong></td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 점수가 얼마나 차이 나는지와 무관하게, 평가용 데이터 정보를 미리 쓴 시점에서 그 평가는 공정하지 않음

> ✅ **한 줄 정리** · 로지스틱에서도 이번에는 점수가 같았지만, 모든 전처리를 Pipeline에 넣고 Pipeline을 train에 fit해야 평가가 공정하다.

**딥러닝 코드에서는**

- `Dataset`의 transform 안에 정규화를 넣고 통계는 train에서만 — Pipeline과 같은 역할

### 5-3. `A` 교차검증으로 누수 vs Pipeline · `cross_val_score` · `StratifiedKFold` <sub>원본 셀 56 ~ 60</sub>

> 🎯 **교차검증에서 누수 scaler와 Pipeline을 비교하면 누수가 더 높게 나오는가?**

💡 **쉽게 말하면**

- Pipeline을 CV에 넣으면 **fold마다** scaler를 학습 부분에서 다시 맞춤.
- 누수 방식은 검증 fold까지 본 기준을 씀.

**실행 흐름**

1. 전체로 fit한 scaler로 미리 변환 (나쁜 예)
2. 그 데이터로 CV
3. Pipeline으로 CV
4. KNN · 로지스틱 둘 다

**먼저 예측** · 누수 방식의 CV 평균이 더 높을까? — 답은 코드 실행 결과 아래에

In [19]:
from sklearn.model_selection import StratifiedKFold, cross_val_score  # 교차검증

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      # 원본과 같은 5등분
X_scaled_bad = StandardScaler().fit(X_iris).transform(X_iris)        # (나쁜 예) 전체로 fit → 변환
knn5 = KNeighborsClassifier(n_neighbors=5)                           # 같은 KNN 설정
lr5000 = LogisticRegression(max_iter=5000, random_state=42)          # 같은 로지스틱 설정
pipe_knn = Pipeline([("scaler", StandardScaler()), ("knn", knn5)])   # fold마다 scaler fit
pipe_lr2 = Pipeline([("scaler", StandardScaler()), ("lr", lr5000)])  # fold마다 scaler fit
rows = [("누수 발생·KNN", knn5, X_scaled_bad),                       # 미리 변환한 데이터
        ("Pipeline·KNN ", pipe_knn, X_iris),                         # 원본 + Pipeline
        ("누수 발생·LR ", lr5000, X_scaled_bad), ("Pipeline·LR  ", pipe_lr2, X_iris)]
for tag, model, data in rows:                                        # 같은 cv로 네 경우
    s = cross_val_score(model, data, y_iris, cv=cv, scoring="accuracy")  # 5-Fold
    print(f"[{tag}] mean={s.mean():.3f} ± {s.std():.3f}")            # 원본 형식

[누수 발생·KNN] mean=0.967 ± 0.037


[Pipeline·KNN ] mean=0.973 ± 0.025
[누수 발생·LR ] mean=0.953 ± 0.045


[Pipeline·LR  ] mean=0.953 ± 0.045


**결과** · 예측 정답은 아님 — KNN은 Pipeline이 오히려 높음 (0.967 vs 0.973)

- `[누수 발생·KNN] mean=0.967 ± 0.037` — 누수 방식
- `[Pipeline·KNN ] mean=0.973 ± 0.025` — 예측과 반대로 Pipeline이 더 높음
- 로지스틱은 똑같음
  - `[누수 발생·LR ] mean=0.953 ± 0.045`
  - `[Pipeline·LR  ] mean=0.953 ± 0.045`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">누수 + CV</td>
<td style="text-align: left;">검증 fold까지 본 기준으로 변환</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">Pipeline + CV</td>
<td style="text-align: left;">fold마다 학습 부분에서 다시 fit</td>
<td style="text-align: left;"><strong>전처리가 있는 CV의 기본</strong></td>
</tr>
<tr>
<td style="text-align: left;">두 점수 차이</td>
<td style="text-align: left;">작거나 반대일 수도 있음</td>
<td style="text-align: left;">판단 기준이 아님</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 누수의 문제는 반드시 점수를 높인다는 것이 아님 → 평가용 데이터를 미리 봐서 공정한 시험이라 부를 수 없다는 것
> - 왜 차이가 작았는지를 억지로 일반화하지 않기

> ✅ **한 줄 정리** · 교차검증에서는 Pipeline이 fold마다 전처리를 다시 맞추며, 누수 방식이 점수가 같거나 낮아도 절차가 틀렸다는 사실은 변하지 않는다.

**딥러닝 코드에서는**

- K-Fold 학습이면 fold마다 정규화 통계를 다시 계산

### 5-4. `A` 난수 데이터 · 없는 패턴이 87% · `SelectKBest` · `f_classif` <sub>원본 셀 61 ~ 63</sub>

> 🎯 **정답과 아무 관계없는 난수에서, 전체로 먼저 특성을 고르면 CV 점수는?**

💡 **쉽게 말하면**

- X는 무작위 숫자, y는 동전 던지기 → 정직하면 **0.5 근처**.
- 전체를 보고 '정답과 관련 있어 보이는' 열 20개를 고르면 정답 정보가 새어 들어감.

**실행 흐름**

1. 100행 × 5000열 난수 · 50 : 50 정답
2. (나쁜 예) 전체로 SelectKBest → CV
3. (올바른 예) Pipeline 안에서 fold마다 선택 → CV

**먼저 예측** · 누수 방식의 CV 정확도는 0.5 근처일까? — 답은 코드 실행 결과 아래에

In [20]:
from sklearn.feature_selection import SelectKBest, f_classif  # 정답과 관련 큰 열 k개 고르기

rng = np.random.RandomState(42)                                 # 원본과 같은 난수
X_rand = rng.normal(size=(100, 5000))                           # 정답과 아무 관계 없는 노이즈
y_rand = np.array([0, 1] * 50)                                  # 정확히 50 : 50
rng.shuffle(y_rand)                                             # 동전 던지기처럼 섞기
selected = SelectKBest(f_classif, k=20).fit_transform(X_rand, y_rand)  # (나쁜 예) 전체로 고름
lr1000 = LogisticRegression(max_iter=1000)                      # 원본 설정
score_leak = cross_val_score(lr1000, selected, y_rand, cv=cv)   # 그다음 CV
pipe_sel = Pipeline([("select", SelectKBest(f_classif, k=20)),  # fold마다 학습 부분에서 선택
                     ("lr", lr1000)])
score_pipe = cross_val_score(pipe_sel, X_rand, y_rand, cv=cv)   # 올바른 CV
print(f"[누수 발생]  mean={score_leak.mean():.3f} +- {score_leak.std():.3f}")  # 원본 형식
print(f"[Pipeline ]  mean={score_pipe.mean():.3f} +- {score_pipe.std():.3f}")  # 정직한 점수
print("[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)")    # 기준

[누수 발생]  mean=0.870 +- 0.081
[Pipeline ]  mean=0.520 +- 0.060
[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)


**결과** · 예측 정답은 아님 — 0.870

- `[누수 발생]  mean=0.870 +- 0.081` — 존재하지 않는 패턴이 87%로 보임 (허구의 성능)
- `[Pipeline ]  mean=0.520 +- 0.060` — 정직한 평가 → 0.5 근처
- `[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)` — 기준

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Iris (5-3)</td>
<td style="text-align: left;">누수하면 점수가 오르나? → 아니오 (0.967 vs 0.973)</td>
<td style="text-align: left;">점수로는 누수를 알 수 없음</td>
</tr>
<tr>
<td style="text-align: left;">난수 (이 섹션)</td>
<td style="text-align: left;">절차만 어기면 안전한가? → 아니오 (0.870 vs 0.520)</td>
<td style="text-align: left;">허구의 성능이 생길 수 있음</td>
</tr>
<tr>
<td style="text-align: left;">판단 기준</td>
<td style="text-align: left;">평가용 데이터 정보를 학습 과정에서 썼는가</td>
<td style="text-align: left;">점수 차이가 아니라 <strong>절차</strong>로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 특성 선택 · PCA · 결측 채우기 · 스케일링 모두 '정답이나 평가용 데이터를 보는' 단계 → 전부 Pipeline 안에

> ✅ **한 줄 정리** · 누수는 어떤 때는 차이가 없고 어떤 때는 없는 패턴을 87%로 만들므로, 점수가 아니라 평가용 데이터를 미리 썼는지로 판단한다.

**딥러닝 코드에서는**

- val 데이터로 어휘 사전 · 정규화 통계를 만들면 같은 누수 → train에서만

---
## 6. 종합 실습 · 지난 숙제 해결 · 펭귄 KNN을 Pipeline으로

> **이 장의 흐름**
>
> - 1 ~ 5장을 한 번에 사용
> - 2회차와 같은 분할에서 스케일러 후보 3개를 Pipeline으로 CV 비교 → 고른 것으로 test 1회

<img src="images/pfm3_practice.png" alt="펭귄 KNN · 스케일링 전후 CV 정확도" width="609">

**파이프라인 위치** · ② 분할 → ③ 전처리(Pipeline) → ⑤ 교차검증 → ⑥ test 1회

> 🎯 **스케일러 후보 3개를 train 안의 CV로 비교하고, 고른 Pipeline으로 test를 한 번 보면?**

💡 **쉽게 말하면**

- 숙제의 원인은 **단위 차이**(3-1) → 스케일러를 Pipeline에 넣어 해결.
- 후보를 정해 CV로 비교하고, test는 마지막에 한 번.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">2회차와 같은 8 : 2 분할 (seed 7)</td>
<td style="text-align: left;">파일별 2 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">스케일러 없음 · Standard · MinMax · Robust Pipeline</td>
<td style="text-align: left;">3장 · 5-1</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">train 안에서 5-Fold CV 비교</td>
<td style="text-align: left;">5-3</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">CV 1등으로 test 1회</td>
<td style="text-align: left;">파일별 2 · 3-3</td>
</tr>
</tbody>
</table>

**먼저 예측** · 스케일링하면 KNN CV가 0.78에서 얼마까지 오를까? — 답은 코드 실행 결과 아래에

In [21]:
# 1단계: 2회차와 같은 분할 (펭귄 342마리 · 3-1의 peng)
Xp, yp = peng[meas].values, peng["species"].to_numpy()  # 측정값 · 품종
# 273 · 69
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.2, stratify=yp, random_state=7)


# 2 · 3단계: 스케일러 후보 4개를 같은 CV로
# 후보 4개 · passthrough = 그대로 통과
cands = {"없음": "passthrough", "Standard": StandardScaler(),
         "MinMax": MinMaxScaler(), "Robust": RobustScaler()}
cv_scores = {}                                              # 후보별 CV 평균
for name, sc_ in cands.items():                             # 후보마다
    # 스케일러만 다름
    pipe_k = Pipeline([("scaler", sc_), ("knn", KNeighborsClassifier(n_neighbors=5))])
    s = cross_val_score(pipe_k, Xp_tr, yp_tr, cv=5)         # 2회차와 같은 cv=5
    cv_scores[name] = s.mean()                              # 평균 저장
    print(f"{name:<9s} CV {s.mean():.4f} ± {s.std():.4f}")  # 후보별 점수

없음        CV 0.7801 ± 0.0355
Standard  CV 0.9854 ± 0.0136


MinMax    CV 0.9854 ± 0.0136
Robust    CV 0.9854 ± 0.0136


In [22]:
# 4단계: CV 1등 스케일러로 test 1회
best_name = max(cv_scores, key=cv_scores.get)               # CV 평균이 가장 높은 후보
# 고른 Pipeline
best_pipe = Pipeline([("scaler", cands[best_name]), ("knn", KNeighborsClassifier(n_neighbors=5))])
# train에 fit · test 1회
test_acc = accuracy_score(yp_te, best_pipe.fit(Xp_tr, yp_tr).predict(Xp_te))
print(f"고른 스케일러: {best_name} · CV {cv_scores[best_name]:.4f} · test {test_acc:.4f}")  # 보고
print(f"2회차 KNN CV 0.7801 → {cv_scores[best_name]:.4f}")  # 숙제 결과

고른 스케일러: Standard · CV 0.9854 · test 1.0000
2회차 KNN CV 0.7801 → 0.9854


**결과** · 예측 정답은 0.9854까지 — 단위만 맞췄을 뿐인데

- `없음      CV 0.7801 ± 0.0355` — 2회차와 같은 값 (숙제의 출발점)
- `Standard  CV 0.9854 ± 0.0136` — 스케일링만으로 0.2 넘게 오름
- `MinMax    CV 0.9854 ± 0.0136` — Standard와 똑같음
- `Robust    CV 0.9854 ± 0.0136` — 셋 다 같음 → 어느 스케일러냐보다 스케일링 여부가 중요
- `고른 스케일러: Standard · CV 0.9854 · test 1.0000` — 동점이면 먼저 넣은 Standard · test 69마리 전부 정답 · 2회차 트리 test 0.9130

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">스케일링 없는 KNN</td>
<td style="text-align: left;">체중이 거리를 지배</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">Pipeline(스케일러 + KNN)</td>
<td style="text-align: left;">fold마다 train에서만 fit</td>
<td style="text-align: left;"><strong>거리 기반 모델의 기본</strong></td>
</tr>
<tr>
<td style="text-align: left;">스케일러 후보 비교</td>
<td style="text-align: left;">CV 평균 · 표준편차로</td>
<td style="text-align: left;">한 번의 분할로 정하지 않음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 세 스케일러가 같은 점수 → 이 데이터에서는 무엇을 쓰든 같음 (항상 그런 것은 아님)
> - test 1.0000을 보고 무언가를 바꾸지 않기 · 69마리 test의 만점은 운도 섞임

> ✅ **한 줄 정리** · KNN의 낮은 점수는 모델 탓이 아니라 단위 차이 탓이었고, 스케일러를 Pipeline에 넣어 CV로 고르면 0.78이 0.985가 된다.

**딥러닝 코드에서는**

- 입력 정규화를 빼먹으면 학습이 안 되거나 느려짐 → `transforms.Normalize`부터 확인

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 인코딩 선택</td>
<td style="text-align: left;">명목형 원-핫 · 순서형 Ordinal(categories=) · LabelEncoder는 주로 y</td>
<td style="text-align: left;">1-1 · 1-5</td>
</tr>
<tr>
<td style="text-align: left;">2. 가짜 순서</td>
<td style="text-align: left;">Label 0.640 = 다수 비율 · 확률 0.6292 &lt; 0.6398 &lt; 0.6503</td>
<td style="text-align: left;">1-2 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">3. 스케일링 · 정규화</td>
<td style="text-align: left;">체중 몫 98.94% · Standard는 -1 ~ 1이 아님 · 정규화는 행 길이 1</td>
<td style="text-align: left;">3-1 · 3-2 · 4-1</td>
</tr>
<tr>
<td style="text-align: left;">4. fit · transform 규칙</td>
<td style="text-align: left;">fit은 train에만 · transform은 둘 다 · Pipeline</td>
<td style="text-align: left;">5-1 · 5-3</td>
</tr>
<tr>
<td style="text-align: left;">5. 누수 판단</td>
<td style="text-align: left;">Iris 0.967 vs 0.973 · 난수 0.870 vs 0.520 → 절차로 판단</td>
<td style="text-align: left;">5-3 · 5-4 · 6</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">원-핫</td>
<td style="text-align: left;">명목형 X의 출발점 · 차원 증가</td>
<td style="text-align: left;"><code>nn.Embedding</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">LabelEncoder</td>
<td style="text-align: left;">주로 y · X에 쓰면 가짜 순서</td>
<td style="text-align: left;">클래스 번호</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">평균 0 · 표준편차 1 · 범위 제한 없음</td>
<td style="text-align: left;"><code>transforms.Normalize</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">fit · transform</td>
<td style="text-align: left;">fit은 train에만 · transform은 둘 다</td>
<td style="text-align: left;">train 통계로 val · test 정규화</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Pipeline</td>
<td style="text-align: left;">전처리 + 모델 · CV에서 fold마다 다시 fit</td>
<td style="text-align: left;"><code>Dataset</code> transform</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">누수</td>
<td style="text-align: left;">점수가 아니라 절차로 판단</td>
<td style="text-align: left;">val로 만든 사전 · 통계 금지</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">Ordinal</td>
<td style="text-align: left;">실제 순서만 · 순서 직접 지정</td>
<td style="text-align: left;">등급 · 학년</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">Robust · MinMax</td>
<td style="text-align: left;">이상치에 강함 · [0, 1]</td>
<td style="text-align: left;">픽셀 0 ~ 1</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">정규화 · 코사인</td>
<td style="text-align: left;">행 길이 1 → 내적 = 코사인</td>
<td style="text-align: left;"><code>F.normalize</code> · 임베딩 검색</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">CountVectorizer</td>
<td style="text-align: left;">사전은 학습 문장으로 · 새 문장은 transform</td>
<td style="text-align: left;">토크나이저 · <code>[UNK]</code></td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">Label과 One-Hot은 각각 무엇을 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">Label 방식 로지스틱이 0.640인 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">레이블 모델의 확률 순서가 뜻하는 것은?</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;">트리는 Label에서도 1.000인 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">1-5</td>
<td style="text-align: left;">OrdinalEncoder에서 <code>categories=</code>를 빼면?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">CountVectorizer 사전은 무엇으로 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">'이 영화는 별로다' 예측의 근거는?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">사전 단어가 겹치면 무엇이 달라지나?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">펭귄 KNN이 낮았던 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">StandardScaler 결과는 -1 ~ 1 안인가?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">이상치가 있으면 어떤 스케일러?</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;">MinMax 0.933 vs Standard 0.911로 결론 낼 수 있나?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">스케일링과 정규화의 기준은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">정규화 내적 = 코사인인 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">코사인 유사도 0.8165의 뜻은?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">fit과 transform의 규칙은?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">점수가 같으면 누수 방식도 괜찮은가?</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">CV에서 Pipeline이 하는 일은?</td>
</tr>
<tr>
<td style="text-align: left;">5-4</td>
<td style="text-align: left;">난수 데이터 누수 결과는?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `[[0, 10], [5, 20], [10, 60]]`을 MinMaxScaler로 바꿔 둘째 열을 출력
2. 같은 배열을 Normalizer로 바꿔 첫 행을 소수 넷째 자리로 출력

In [23]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 1-2에서 정답 규칙을 'blue면 1, 나머지 0'으로 바꾸면 Label 방식 로지스틱 정확도는? (blue가 번호 끝)
2. 5-4에서 `k=20`을 `k=200`으로 바꾸면 누수 방식 점수는?
3. 6장에서 `n_neighbors=15`로 바꾸면 스케일링 없음 · Standard의 CV 평균은?

In [24]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">번호 하나 (순서 생김) vs 카테고리마다 0 / 1 열 (순서 없음 · 차원 증가)</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">green이 번호 가운데라 직선으로 못 가름 → 모두 1 예측 = 다수 비율</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">모델이 임의의 번호를 진짜 순서로 믿고 그대로 씀</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;">기준값을 여러 번(≤ 0.5 · ≤ 1.5) 둘 수 있어서 · 그래도 순서를 쓰는 것은 같음</td>
</tr>
<tr>
<td style="text-align: left;">1-5</td>
<td style="text-align: left;">알파벳 순이 되어 실제 순서가 틀어짐</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">학습 문장(fit) · 한 글자 단어는 기본 설정에서 빠짐</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">없음 · 매칭 단어 0개 · 확률 [0.5, 0.5] · 버전에 따라 0 / 1이 뒤집힘</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">가중치가 근거가 되어 확률이 반반에서 움직임 (0.5728)</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">체중 범위 3600이 거리의 98.94%를 차지 → 체중만 봄</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">아님 · 평균 0 · 표준편차 1일 뿐 (1.305)</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">RobustScaler (중앙값 · IQR)</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;">없음 · 1개 차이 · 한 번의 분할 → 후보를 CV로 비교</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">스케일링은 열 · 정규화는 행</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">길이가 1이라 분모가 사라짐</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">단어 사용 패턴의 방향이 비슷 · 의미가 같다는 뜻 아님</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">fit은 train에만 · transform은 train · test 둘 다</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">아님 · 평가용 데이터를 미리 쓴 시점에서 불공정</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">fold마다 학습 부분에서 전처리를 다시 fit</td>
</tr>
<tr>
<td style="text-align: left;">5-4</td>
<td style="text-align: left;">누수 0.870 vs Pipeline 0.520 (정답 0.5) · 없는 패턴이 보임</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
arr = np.array([[0, 10], [5, 20], [10, 60]])
print(MinMaxScaler().fit_transform(arr)[:, 1].tolist())  # [0.0, 0.2, 1.0]
print(Normalizer().fit_transform(arr)[0].round(4).tolist())  # [0.0, 1.0]
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">1.000 — blue가 번호 0(끝)이라 직선 하나로 가를 수 있음 · 가짜 순서가 맞아떨어진 우연 (blue 비율 0.31)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">1.000 — 고른 열이 많을수록 우연히 맞는 패턴이 더 많이 새어 들어감 (Pipeline은 0.5 근처)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">없음 0.758 · Standard 0.9818 — k를 바꿔도 스케일링 차이가 훨씬 큼</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 누수 방식 점수가 같으니 괜찮다고 답함</td>
<td style="text-align: left;">점수로 누수를 판단</td>
<td style="text-align: left;">평가용 데이터 정보를 썼는지 절차로 판단</td>
<td style="text-align: left;">fit에 test(또는 검증 fold)가 들어갔나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 원-핫 인코딩과 레이블 인코딩은 언제 쓰나요?**
> - 순서가 없는 명목형 입력은 원-핫이 기본이고, 레이블 인코딩은 주로 타깃 y에 씁니다.
> - 명목형 입력에 번호를 붙이면 선형 모델이 그 번호를 크기로 써서, 색깔 예제에서는 학습 정확도가 다수 비율 0.640에 머물렀습니다.
> - 실제 순서가 있는 범주는 OrdinalEncoder에 순서를 직접 지정합니다.

**Q2. 스케일링은 왜 하고, 정규화와는 무엇이 다른가요?**
> - KNN처럼 거리를 쓰는 모델은 단위가 큰 특성이 거리를 지배합니다.
> - 펭귄에서는 체중이 거리의 98.94%를 차지했고, StandardScaler를 넣자 CV가 0.78에서 0.985로 올랐습니다.
> - 스케일링은 열 기준으로 크기를 맞추고, 정규화는 행 기준으로 길이를 1로 맞춰 방향만 비교합니다.

**Q3. 데이터 누수는 어떻게 막고, 어떻게 판단하나요?**
> - 전처리의 fit은 train에만 하고 transform은 train과 test 모두에 적용하며, Pipeline으로 묶어 교차검증에서도 fold마다 다시 fit되게 합니다.
> - 누수는 점수 차이로 판단하지 않습니다.
> - Iris에서는 점수가 비슷했지만 난수 데이터에서는 누수 방식이 없는 패턴을 0.87로 보이게 했습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>LabelEncoder</code> · <code>OneHotEncoder</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">번호 · 원-핫</td>
<td style="text-align: left;">1-1 <code>ohe.fit_transform(data[["color"]])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>OrdinalEncoder(categories=)</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">순서형 번호</td>
<td style="text-align: left;">1-5 <code>OrdinalEncoder(categories=[[…]])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>coef_</code> · <code>predict_proba</code></td>
<td style="text-align: left;">LogisticRegression</td>
<td style="text-align: left;">모델 열어 보기</td>
<td style="text-align: left;">1-3 <code>lr_label.predict_proba([[v]])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>CountVectorizer</code></td>
<td style="text-align: left;">sklearn.feature_extraction.text</td>
<td style="text-align: left;">단어 빈도 벡터</td>
<td style="text-align: left;">2-1 <code>vectorizer.fit_transform(texts)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>transform</code> · <code>nnz</code></td>
<td style="text-align: left;">CountVectorizer · 희소 행렬</td>
<td style="text-align: left;">새 문장 변환 · 0 아닌 칸 수</td>
<td style="text-align: left;">2-2 <code>new_X.nnz</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>StandardScaler</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">평균 0 · 표준편차 1</td>
<td style="text-align: left;">3-2 <code>scaler.fit_transform(X_money)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>RobustScaler</code> · <code>MinMaxScaler</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">중앙값 · IQR · [0, 1]</td>
<td style="text-align: left;">3-3 <code>RobustScaler().fit_transform(X_out)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>Normalizer</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">행 길이 1</td>
<td style="text-align: left;">4-1 <code>Normalizer().fit_transform(X_vec)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>cosine_similarity</code></td>
<td style="text-align: left;">sklearn.metrics.pairwise</td>
<td style="text-align: left;">방향 유사도</td>
<td style="text-align: left;">4-2 <code>cosine_similarity(A, B)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>Pipeline</code></td>
<td style="text-align: left;">sklearn.pipeline</td>
<td style="text-align: left;">전처리 + 모델 · fit 범위 관리</td>
<td style="text-align: left;">5-1 <code>Pipeline([("scaler", …), ("knn", …)])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>cross_val_score</code> · <code>StratifiedKFold</code></td>
<td style="text-align: left;">sklearn.model_selection</td>
<td style="text-align: left;">교차검증</td>
<td style="text-align: left;">5-3 <code>cross_val_score(model, data, y_iris, cv=cv)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>SelectKBest(f_classif)</code></td>
<td style="text-align: left;">sklearn.feature_selection</td>
<td style="text-align: left;">정답과 관련 큰 열 고르기</td>
<td style="text-align: left;">5-4 <code>SelectKBest(f_classif, k=20)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">명목형 X는 원-핫</td>
<td style="text-align: left;">명목형 X에 LabelEncoder</td>
</tr>
<tr>
<td style="text-align: left;">순서형은 <code>categories=</code>로 순서 지정</td>
<td style="text-align: left;">Standard 결과를 -1 ~ 1로 착각</td>
</tr>
<tr>
<td style="text-align: left;">새 문장 · test는 transform만</td>
<td style="text-align: left;">새 데이터에 fit_transform</td>
</tr>
<tr>
<td style="text-align: left;">전처리는 Pipeline 안에</td>
<td style="text-align: left;">전체 데이터로 scaler · 특성 선택 fit</td>
</tr>
<tr>
<td style="text-align: left;">누수는 절차로 판단</td>
<td style="text-align: left;">점수가 같으니 괜찮다고 판단</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Label vs One-Hot</td>
<td style="text-align: left;">번호 하나 vs 0 / 1 열</td>
<td style="text-align: left;">X는 원-핫, y는 Label</td>
</tr>
<tr>
<td style="text-align: left;">스케일링 vs 정규화</td>
<td style="text-align: left;">열 기준 vs 행 기준</td>
<td style="text-align: left;">거리 · 선형은 스케일링, 방향 비교는 정규화</td>
</tr>
<tr>
<td style="text-align: left;">Standard vs Robust</td>
<td style="text-align: left;">평균 · 표준편차 vs 중앙값 · IQR</td>
<td style="text-align: left;">이상치가 있으면 Robust</td>
</tr>
<tr>
<td style="text-align: left;"><code>fit</code> vs <code>transform</code></td>
<td style="text-align: left;">기준 배우기 vs 기준 적용</td>
<td style="text-align: left;">fit은 train에만</td>
</tr>
<tr>
<td style="text-align: left;">내적 vs 코사인</td>
<td style="text-align: left;">길이 포함 vs 방향만</td>
<td style="text-align: left;">정규화하면 같아짐</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 · 1-2 확률 세 줄이 왜 생기는지 말로 설명</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">6장 종합 실습을 빈 셀에 처음부터 · 5-4 난수 실험을 k를 바꿔 다시</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 4(평가지표) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 셀 66개를 원본 Part 순서대로 1 ~ 5장 · 18개 섹션으로 나눔 (인코딩을 범주 · 텍스트 두 장으로)
- 원본의 펭귄 KNN 숙제(셀 27)를 6장 종합 실습에서 실제로 풀어 봄 (새로 추가)
- 원본 비유 그림(스크린샷 2장)은 0장 힌트의 글로 · 원본 표 · 선택 가이드는 섹션별 '차이 · 언제 쓰나' 표로
- 원본 셀 29 주석의 '-1 ~ 1'은 틀린 설명이라 3-2에서 바로잡음 · 모든 코드는 실행한 출력과 함께 · 형식 v2.1

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 0 · 1</td>
<td style="text-align: left;">목표 · 전처리가 필요한 이유 · 셰프 그림 2장</td>
<td style="text-align: left;">0. 원본 목표 · 힌트</td>
</tr>
<tr>
<td style="text-align: left;">셀 2 ~ 7</td>
<td style="text-align: left;">인코딩 · Label vs One-Hot · 차원</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 8 ~ 11</td>
<td style="text-align: left;">레이블 인코딩의 문제 (선형 모델)</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 12 · 13</td>
<td style="text-align: left;">모델을 열어 보기 · 확률 순서</td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 14</td>
<td style="text-align: left;">결정 트리</td>
<td style="text-align: left;">1-4</td>
</tr>
<tr>
<td style="text-align: left;">셀 15 ~ 18</td>
<td style="text-align: left;">Ordinal · 인코딩 비교표 · 선택 가이드</td>
<td style="text-align: left;">1-5</td>
</tr>
<tr>
<td style="text-align: left;">셀 19 · 20</td>
<td style="text-align: left;">CountVectorizer</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 21 ~ 25</td>
<td style="text-align: left;">새 문장 · 출력 ≠ 근거</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 26</td>
<td style="text-align: left;">사전 단어로</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 27</td>
<td style="text-align: left;">지난 숙제 (펭귄 KNN)</td>
<td style="text-align: left;">3-1 · 6. 종합 실습</td>
</tr>
<tr>
<td style="text-align: left;">셀 28 ~ 30</td>
<td style="text-align: left;">StandardScaler</td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 31 · 32</td>
<td style="text-align: left;">RobustScaler</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 33 ~ 37</td>
<td style="text-align: left;">MinMax vs Standard · 스케일러 비교 · 출발점</td>
<td style="text-align: left;">3-4</td>
</tr>
<tr>
<td style="text-align: left;">셀 38 · 39</td>
<td style="text-align: left;">정규화</td>
<td style="text-align: left;">4-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 40 ~ 42</td>
<td style="text-align: left;">정규화와 코사인</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 43 ~ 45</td>
<td style="text-align: left;">문장 유사도 · 스케일링 vs 정규화</td>
<td style="text-align: left;">4-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 46 ~ 52</td>
<td style="text-align: left;">뻥튀기 복습 · 누수 · fit/transform 규칙 · Pipeline</td>
<td style="text-align: left;">5-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 53 ~ 55</td>
<td style="text-align: left;">로지스틱 · 결과 해석</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 56 ~ 60</td>
<td style="text-align: left;">교차검증으로 누수 vs Pipeline</td>
<td style="text-align: left;">5-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 61 ~ 63</td>
<td style="text-align: left;">난수 데이터 · 두 장면</td>
<td style="text-align: left;">5-4</td>
</tr>
<tr>
<td style="text-align: left;">셀 64 · 65</td>
<td style="text-align: left;">오늘의 정리 · 관통 원칙 · 다음 시간 예고</td>
<td style="text-align: left;">그래서 오늘 · 최종 요약표 · 다음 복습일</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 6장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [25]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # 글자 → 숫자
from sklearn.linear_model import LogisticRegression            # 선형 모델
from sklearn.metrics import accuracy_score                     # 정확도
from sklearn.tree import DecisionTreeClassifier                # 기준값으로 나누는 모델
from sklearn.preprocessing import OrdinalEncoder               # 순서형 → 번호


# ---------- 1-1 ----------

data = pd.DataFrame({"color": ["red", "green", "blue", "red", "green", "blue"]})  # 글자 열 1개
le = LabelEncoder()                                            # 번호 붙이기
data["color_label"] = le.fit_transform(data["color"])          # 0 · 1 · 2 번호 (라이브러리가 정함)
ohe = OneHotEncoder(sparse_output=False)                       # 배열로 받기
ohe_matrix = ohe.fit_transform(data[["color"]])                # 카테고리마다 새 열 · 행 수는 그대로
ohe_df = pd.DataFrame(ohe_matrix, columns=ohe.categories_[0])  # 열 이름 = 카테고리
data_ohe = pd.concat([data, ohe_df], axis=1)                   # 옆으로 합치기
print(data_ohe.to_string())                                    # 원본 셀 6의 표
# 열 개수 = 차원
print("Label 순서:", list(le.classes_), "→ 차원 1 → One-Hot 차원", ohe_matrix.shape[1])


# ---------- 1-2 ----------

np.random.seed(42)                                                               # 원본과 같은 난수
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))         # 색 100개
target = colors.map({"red": 1, "blue": 1, "green": 0})  # red · blue → 1, green → 0
df_test = pd.DataFrame({"color": colors, "target": target})                      # 실험용 표
X_label = LabelEncoder().fit_transform(df_test["color"]).reshape(-1, 1)          # 번호 열 1개
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(df_test[["color"]])  # 0 / 1 열 3개
y = df_test["target"]                                                            # 정답
# 같은 모델 · 표현만 다름
acc_label = accuracy_score(y, LogisticRegression().fit(X_label, y).predict(X_label))
acc_onehot = accuracy_score(y, LogisticRegression().fit(X_onehot, y).predict(X_onehot))  # 같은 모델
print("=== 로지스틱 회귀 (선형 모델) ===")                                       # 원본 출력 형식
print(f"레이블 방식 학습 데이터 정확도: {acc_label:.3f}")                        # 번호
print(f"원-핫 방식 학습 데이터 정확도: {acc_onehot:.3f}")                        # 원-핫
print(f"참고 - 다수 클래스 비율: {y.mean():.3f}")                                # 1의 비율


# ---------- 1-3 ----------
lr_label = LogisticRegression().fit(X_label, y)                           # 레이블 방식 모델
print(f"coef_      : {lr_label.coef_[0][0]:.4f}")                         # 번호 1 늘 때 점수 변화
print(f"intercept_ : {lr_label.intercept_[0]:.4f}")                       # 번호 0일 때 점수
print(f"n_iter_    : {lr_label.n_iter_[0]}  (수렴 완료)")                 # 학습은 끝까지 됨
print("입력 숫자별 예측 확률 P(target=1)")                                # 번호마다
for v, name in enumerate(LabelEncoder().fit(df_test["color"]).classes_):  # 0 blue · 1 green · 2 red
    p = lr_label.predict_proba([[v]])[0, 1]                               # 1일 확률
    print(f"  {name:6s}({v}) -> {p:.4f}")                                 # 번호 순으로 커지는지


# ---------- 1-4 ----------

dt_label = DecisionTreeClassifier(random_state=42).fit(X_label, y)    # 번호 열
dt_onehot = DecisionTreeClassifier(random_state=42).fit(X_onehot, y)  # 원-핫 열
print("=== 결정 트리 (트리 모델) ===")                                # 원본 출력 형식
print(f"레이블 방식 학습 데이터 정확도: {accuracy_score(y, dt_label.predict(X_label)):.3f}")  # 번호
# 원-핫
print(f"원-핫 방식 학습 데이터 정확도: {accuracy_score(y, dt_onehot.predict(X_onehot)):.3f}")
# 질문 횟수
print("레이블 트리 깊이:", dt_label.get_depth(), "· 원-핫 트리 깊이:", dt_onehot.get_depth())

   color  color_label  blue  green  red
0    red            2   0.0    0.0  1.0
1  green            1   0.0    1.0  0.0
2   blue            0   1.0    0.0  0.0
3    red            2   0.0    0.0  1.0
4  green            1   0.0    1.0  0.0
5   blue            0   1.0    0.0  0.0
Label 순서: ['blue', 'green', 'red'] → 차원 1 → One-Hot 차원 3
=== 로지스틱 회귀 (선형 모델) ===
레이블 방식 학습 데이터 정확도: 0.640
원-핫 방식 학습 데이터 정확도: 1.000
참고 - 다수 클래스 비율: 0.640
coef_      : 0.0458
intercept_ : 0.5288
n_iter_    : 4  (수렴 완료)
입력 숫자별 예측 확률 P(target=1)
  blue  (0) -> 0.6292
  green (1) -> 0.6398
  red   (2) -> 0.6503
=== 결정 트리 (트리 모델) ===
레이블 방식 학습 데이터 정확도: 1.000
원-핫 방식 학습 데이터 정확도: 1.000
레이블 트리 깊이: 2 · 원-핫 트리 깊이: 1


In [26]:
# ---------- 1-5 ----------

df_ord = pd.DataFrame({"size": ["small", "medium", "large", "medium", "small"]})  # 순서 있는 범주
ord_enc = OrdinalEncoder(categories=[["small", "medium", "large"]])  # 순서를 직접 지정 → 0 · 1 · 2
df_ord["size_code"] = ord_enc.fit_transform(df_ord[["size"]])        # 학습 + 변환 한 번에
print(df_ord.to_string())                                            # 원본 셀 16의 표
print("지정 안 하면:", OrdinalEncoder().fit(df_ord[["size"]]).categories_[0])  # 알파벳 순이 됨

     size  size_code
0   small        0.0
1  medium        1.0
2   large        2.0
3  medium        1.0
4   small        0.0
지정 안 하면: ['large' 'medium' 'small']


In [27]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터


# ---------- 2-1 ----------

texts = ["이 영화 정말 재밌다", "완전 감동적이다", "너무 지루하고 별로였다",  # 문장 6개
         "시간 낭비였다", "연출이 좋았다", "스토리가 엉망이다"]
labels = [1, 1, 0, 0, 1, 0]                                    # 1 = 긍정, 0 = 부정
vectorizer = CountVectorizer()                                 # 공백으로 나눠 단어를 셈
X_text = vectorizer.fit_transform(texts)                       # 사전 만들기 + 벡터화
print("단어 사전:", list(vectorizer.get_feature_names_out()))  # 원본 셀 20
print("벡터 모양:", X_text.shape)                              # (문장 수, 사전 크기)
text_model = LogisticRegression().fit(X_text, labels)          # 감성 분류기 (toy)


# ---------- 2-2 ----------
new_X = vectorizer.transform(["이 영화는 별로다"])  # 사전은 그대로 · transform만
feature_names = vectorizer.get_feature_names_out()                              # 사전 단어
print("예측 결과:", text_model.predict(new_X))                                  # 원본 셀 22
print("사전에서 매칭된 단어 수:", new_X.nnz, "개")                              # 0이 아닌 칸 수
print("매칭된 단어           :", list(feature_names[new_X.toarray()[0] > 0]))   # 겹친 단어
print("예측 확률 [부정, 긍정] :", text_model.predict_proba(new_X)[0].round(4))  # 근거가 있나


# ---------- 2-3 ----------
better_X = vectorizer.transform(["정말 지루하고 별로였다"])                        # 사전 단어 3개
print("사전에서 매칭된 단어 수:", better_X.nnz, "개")                              # 겹친 단어 수
print("매칭된 단어           :", list(feature_names[better_X.toarray()[0] > 0]))   # 어떤 단어
print("예측 확률 [부정, 긍정] :", text_model.predict_proba(better_X)[0].round(4))  # 근거가 생김
print("예측 결과             :", text_model.predict(better_X)[0])                  # 부정

단어 사전: ['감동적이다', '낭비였다', '너무', '별로였다', '스토리가', '시간', '엉망이다', '연출이', '영화', '완전', '재밌다', '정말', '좋았다', '지루하고']
벡터 모양: (6, 14)
예측 결과: [1]
사전에서 매칭된 단어 수: 0 개
매칭된 단어           : []
예측 확률 [부정, 긍정] : [0.5 0.5]
사전에서 매칭된 단어 수: 3 개
매칭된 단어           : ['별로였다', '정말', '지루하고']
예측 확률 [부정, 긍정] : [0.5728 0.4272]
예측 결과             : 0


In [28]:
from sklearn.preprocessing import StandardScaler      # 평균 0 · 표준편차 1
from sklearn.preprocessing import RobustScaler        # 중앙값 · IQR 기반
from sklearn.datasets import load_iris                # Iris
from sklearn.metrics import f1_score                  # 클래스별 F1의 평균(macro)
from sklearn.model_selection import train_test_split  # 분할
from sklearn.neighbors import KNeighborsClassifier    # 거리 기반 모델
from sklearn.pipeline import Pipeline                 # 전처리 + 모델 묶기
from sklearn.preprocessing import MinMaxScaler        # [0, 1]


# ---------- 3-1 ----------
CSV = "../02_모델선택_펭귄/penguins.csv"                            # 원본 폴더 데이터
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]  # 측정값 4개
peng = pd.read_csv(CSV).dropna(subset=meas).reset_index(drop=True)  # 2회차와 같은 342마리
rng_tbl = peng[meas].agg(["min", "max"]).T                          # 열마다 최소 · 최대
rng_tbl["범위"] = rng_tbl["max"] - rng_tbl["min"]                   # 크기 차이
print(rng_tbl.round(1).to_string())                                 # 범위 비교
d = (peng.loc[0, meas] - peng.loc[1, meas]).astype(float)           # 첫 두 마리의 차이
share = (d ** 2) / (d ** 2).sum()                                   # 거리²에서 열마다 몫
print("거리 몫:", share.round(4).to_dict())                         # 체중이 거의 전부


# ---------- 3-2 ----------

X_money = np.array([[150, 1000], [160, 20000], [170, 50000]])  # 키(cm) · 용돈(원)
scaler = StandardScaler()                                      # 열마다 따로 계산
X_scaled = scaler.fit_transform(X_money)                       # 평균 · 표준편차를 배우고 변환
df_scale = pd.DataFrame(np.hstack([X_money, X_scaled]),        # 전후를 옆으로
                        columns=["키(cm)", "용돈(원)", "키(스케일링 후)", "용돈(스케일링 후)"])
print(df_scale.round(6).to_string())                           # 원본 셀 29의 표
# 배운 값
print("열 평균:", scaler.mean_.round(1).tolist(), "· 열 표준편차:", scaler.scale_.round(1).tolist())


# ---------- 3-3 ----------

X_out = np.array([[1., 10.], [2., 12.], [3., 13.], [100., 1000.]])  # 마지막 행이 이상치
sc = StandardScaler().fit_transform(X_out)                 # 평균 · 표준편차 → 이상치에 민감
rb = RobustScaler().fit_transform(X_out)                   # 중앙값 · IQR → 덜 민감
df_rb = pd.DataFrame(np.hstack([X_out, sc, rb]),           # 전후를 옆으로
                     columns=["x1", "x2", "Std_x1", "Std_x2", "Robust_x1", "Robust_x2"])
print(df_rb.round(6).to_string())                          # 원본 셀 32의 표
gap_sc, gap_rb = sc[2, 0] - sc[0, 0], rb[2, 0] - rb[0, 0]  # 정상 값 1과 3 사이 간격
print("정상 3개 폭 · Std:", round(gap_sc, 3), "· Robust:", round(gap_rb, 3))  # 간격 비교


# ---------- 3-4 ----------

X_iris, y_iris = load_iris(return_X_y=True)                        # 150 × 4
X_train, X_test, y_train, y_test = train_test_split(               # 원본과 같은 7 : 3
    X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=42)
# MinMax
pipe_minmax = Pipeline([("scaler", MinMaxScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
# Standard
pipe_std = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
for tag, pipe in [("MinMax", pipe_minmax), ("Std   ", pipe_std)]:  # 같은 코드로 둘 다
    pred = pipe.fit(X_train, y_train).predict(X_test)              # train에 fit · test 예측
    acc, f1 = accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")  # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")            # 원본 형식

                      min     max      범위
bill_length_mm       32.1    59.6    27.5
bill_depth_mm        13.1    21.5     8.4
flipper_length_mm   172.0   231.0    59.0
body_mass_g        2700.0  6300.0  3600.0


거리 몫: {'bill_length_mm': 0.0001, 'bill_depth_mm': 0.0007, 'flipper_length_mm': 0.0099, 'body_mass_g': 0.9894}
   키(cm)    용돈(원)  키(스케일링 후)  용돈(스케일링 후)
0  150.0   1000.0  -1.224745   -1.123698
1  160.0  20000.0   0.000000   -0.181775
2  170.0  50000.0   1.224745    1.305473
열 평균: [160.0, 23666.7] · 열 표준편차: [8.2, 20171.5]
      x1      x2    Std_x1    Std_x2  Robust_x1  Robust_x2
0    1.0    10.0 -0.600832 -0.581243  -0.058824  -0.010070
1    2.0    12.0 -0.577270 -0.576570  -0.019608  -0.002014
2    3.0    13.0 -0.553708 -0.574233   0.019608   0.002014
3  100.0  1000.0  1.731810  1.732045   3.823529   3.977845
정상 3개 폭 · Std: 0.047 · Robust: 0.078
[MinMax] Acc=0.933, F1(macro)=0.933
[Std   ] Acc=0.911, F1(macro)=0.910


In [29]:
from sklearn.preprocessing import Normalizer            # 행(샘플)마다 길이 1
from sklearn.metrics.pairwise import cosine_similarity  # 방향 유사도


# ---------- 4-1 ----------

X_vec = np.array([[3, 4], [1, 2], [10, 0]])                         # 샘플 3개
X_norm = Normalizer().fit_transform(X_vec)                          # 행마다 길이로 나눔 (L2)
cols = ["x1", "x2", "x1(정규화 후)", "x2(정규화 후)"]               # 원본 열 이름
df_norm = pd.DataFrame(np.hstack([X_vec, X_norm]), columns=cols)    # 전후를 옆으로
print(df_norm.round(6).to_string())                                 # 원본 셀 39의 표
print("행 길이:", np.linalg.norm(X_norm, axis=1).round(3))          # 전부 1
music = Normalizer().fit_transform(np.array([[100, 50], [10, 5]]))  # 록 · 재즈 곡 수
print("취향 A · B 정규화:", music.round(4).tolist())                # 같은 방향


# ---------- 4-2 ----------

A = np.array([[3., 4.]])                                                 # 길이 5
B = np.array([[8., 6.]])  # 길이 10 · 평행하지 않음 (평행하면 코사인이 항상 1)
normer = Normalizer()                                                    # 길이 1로
A_n, B_n = normer.fit_transform(A), normer.transform(B)                  # 행마다 따로 나눔
print(f"원래 벡터의 내적        : {float(np.dot(A[0], B[0])):.4f}")      # 길이 · 방향 둘 다 반영
print(f"정규화된 벡터의 내적    : {float(np.dot(A_n[0], B_n[0])):.4f}")  # 방향만
print(f"cosine_similarity      : {float(cosine_similarity(A, B)[0, 0]):.4f}")  # 방향만


# ---------- 4-3 ----------
pair = CountVectorizer().fit(["영화가 좋다", "영화가 진짜로 좋다"])            # 두 문장 사전
V = pair.transform(["영화가 좋다", "영화가 진짜로 좋다"])                      # 단어 빈도 벡터
print("단어 사전:", list(pair.get_feature_names_out()))                        # 단어 3개
print("코사인 유사도:", round(float(cosine_similarity(V[0], V[1])[0, 0]), 4))  # 안에서 정규화함
W = pair.transform(["영화가 좋다", "영화가 별로다"])  # 반대 뜻 문장 ('별로다'는 사전에 없음)
print("반대 뜻 문장 유사도:", round(float(cosine_similarity(W[0], W[1])[0, 0]), 4))  # 의미와 무관

     x1   x2  x1(정규화 후)  x2(정규화 후)
0   3.0  4.0   0.600000   0.800000
1   1.0  2.0   0.447214   0.894427
2  10.0  0.0   1.000000   0.000000
행 길이: [1. 1. 1.]
취향 A · B 정규화: [[0.8944, 0.4472], [0.8944, 0.4472]]
원래 벡터의 내적        : 48.0000
정규화된 벡터의 내적    : 0.9600
cosine_similarity      : 0.9600
단어 사전: ['영화가', '좋다', '진짜로']
코사인 유사도: 0.8165
반대 뜻 문장 유사도: 0.7071


In [30]:
from sklearn.model_selection import StratifiedKFold, cross_val_score  # 교차검증
from sklearn.feature_selection import SelectKBest, f_classif          # 정답과 관련 큰 열 k개 고르기


# ---------- 5-1 ----------
scaler_bad = StandardScaler().fit(X_iris)                # (나쁜 예) test까지 포함한 평균 · 표준편차
X_train_bad = scaler_bad.transform(X_train)              # 누수된 기준으로 변환
X_test_bad = scaler_bad.transform(X_test)                # 같은 기준
knn_bad = KNeighborsClassifier(n_neighbors=5).fit(X_train_bad, y_train)  # 누수 데이터로 KNN
pred_bad = knn_bad.predict(X_test_bad)                   # 누수 기준의 test 예측
pipe = Pipeline([("scaler", StandardScaler()),           # (올바른 예) train에만 fit
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
pred_pipe = pipe.fit(X_train, y_train).predict(X_test)   # scaler도 train에서만 fit
for tag, p in [("누수 발생·KNN", pred_bad), ("Pipeline·KNN ", pred_pipe)]:  # 같은 형식으로
    acc, f1 = accuracy_score(y_test, p), f1_score(y_test, p, average="macro")  # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")  # 원본 형식
mean_gap = scaler_bad.mean_ - pipe.named_steps["scaler"].mean_  # 두 scaler가 배운 평균의 차이
print("평균 차이(누수 - train):", mean_gap.round(3))     # test 정보만큼 기준이 다름


# ---------- 5-2 ----------
lr_bad = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_bad, y_train)  # 누수 데이터
pred_lr_bad = lr_bad.predict(X_test_bad)                                        # 누수 기준의 test
pipe_lr = Pipeline([("scaler", StandardScaler()),                               # train에만 fit
                    ("lr", LogisticRegression(max_iter=5000, random_state=42))])
pred_lr_pipe = pipe_lr.fit(X_train, y_train).predict(X_test)                    # 올바른 절차
for tag, p in [("누수 발생·LR", pred_lr_bad), ("Pipeline·LR ", pred_lr_pipe)]:  # 같은 형식
    acc, f1 = accuracy_score(y_test, p), f1_score(y_test, p, average="macro")   # 두 지표
    print(f"[{tag}] Acc={acc:.3f}, F1(macro)={f1:.3f}")                         # 원본 형식


# ---------- 5-3 ----------

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      # 원본과 같은 5등분
X_scaled_bad = StandardScaler().fit(X_iris).transform(X_iris)        # (나쁜 예) 전체로 fit → 변환
knn5 = KNeighborsClassifier(n_neighbors=5)                           # 같은 KNN 설정
lr5000 = LogisticRegression(max_iter=5000, random_state=42)          # 같은 로지스틱 설정
pipe_knn = Pipeline([("scaler", StandardScaler()), ("knn", knn5)])   # fold마다 scaler fit
pipe_lr2 = Pipeline([("scaler", StandardScaler()), ("lr", lr5000)])  # fold마다 scaler fit
rows = [("누수 발생·KNN", knn5, X_scaled_bad),                       # 미리 변환한 데이터
        ("Pipeline·KNN ", pipe_knn, X_iris),                         # 원본 + Pipeline
        ("누수 발생·LR ", lr5000, X_scaled_bad), ("Pipeline·LR  ", pipe_lr2, X_iris)]
for tag, model, data in rows:                                        # 같은 cv로 네 경우
    s = cross_val_score(model, data, y_iris, cv=cv, scoring="accuracy")  # 5-Fold
    print(f"[{tag}] mean={s.mean():.3f} ± {s.std():.3f}")            # 원본 형식

[누수 발생·KNN] Acc=0.911, F1(macro)=0.910
[Pipeline·KNN ] Acc=0.911, F1(macro)=0.910
평균 차이(누수 - train): [-0.03   0.002 -0.027 -0.006]
[누수 발생·LR] Acc=0.911, F1(macro)=0.911
[Pipeline·LR ] Acc=0.911, F1(macro)=0.911


[누수 발생·KNN] mean=0.967 ± 0.037


[Pipeline·KNN ] mean=0.973 ± 0.025


[누수 발생·LR ] mean=0.953 ± 0.045

[Pipeline·LR  ] mean=0.953 ± 0.045


In [31]:
# ---------- 5-4 ----------

rng = np.random.RandomState(42)                                 # 원본과 같은 난수
X_rand = rng.normal(size=(100, 5000))                           # 정답과 아무 관계 없는 노이즈
y_rand = np.array([0, 1] * 50)                                  # 정확히 50 : 50
rng.shuffle(y_rand)                                             # 동전 던지기처럼 섞기
selected = SelectKBest(f_classif, k=20).fit_transform(X_rand, y_rand)  # (나쁜 예) 전체로 고름
lr1000 = LogisticRegression(max_iter=1000)                      # 원본 설정
score_leak = cross_val_score(lr1000, selected, y_rand, cv=cv)   # 그다음 CV
pipe_sel = Pipeline([("select", SelectKBest(f_classif, k=20)),  # fold마다 학습 부분에서 선택
                     ("lr", lr1000)])
score_pipe = cross_val_score(pipe_sel, X_rand, y_rand, cv=cv)   # 올바른 CV
print(f"[누수 발생]  mean={score_leak.mean():.3f} +- {score_leak.std():.3f}")  # 원본 형식
print(f"[Pipeline ]  mean={score_pipe.mean():.3f} +- {score_pipe.std():.3f}")  # 정직한 점수
print("[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)")    # 기준

[누수 발생]  mean=0.870 +- 0.081
[Pipeline ]  mean=0.520 +- 0.060
[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)


In [32]:
# ---------- 6. 종합 실습 ----------
# 1단계: 2회차와 같은 분할 (펭귄 342마리 · 3-1의 peng)
Xp, yp = peng[meas].values, peng["species"].to_numpy()  # 측정값 · 품종
# 273 · 69
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.2, stratify=yp, random_state=7)


# 2 · 3단계: 스케일러 후보 4개를 같은 CV로
# 후보 4개 · passthrough = 그대로 통과
cands = {"없음": "passthrough", "Standard": StandardScaler(),
         "MinMax": MinMaxScaler(), "Robust": RobustScaler()}
cv_scores = {}                                              # 후보별 CV 평균
for name, sc_ in cands.items():                             # 후보마다
    # 스케일러만 다름
    pipe_k = Pipeline([("scaler", sc_), ("knn", KNeighborsClassifier(n_neighbors=5))])
    s = cross_val_score(pipe_k, Xp_tr, yp_tr, cv=5)         # 2회차와 같은 cv=5
    cv_scores[name] = s.mean()                              # 평균 저장
    print(f"{name:<9s} CV {s.mean():.4f} ± {s.std():.4f}")  # 후보별 점수


# 4단계: CV 1등 스케일러로 test 1회
best_name = max(cv_scores, key=cv_scores.get)               # CV 평균이 가장 높은 후보
# 고른 Pipeline
best_pipe = Pipeline([("scaler", cands[best_name]), ("knn", KNeighborsClassifier(n_neighbors=5))])
# train에 fit · test 1회
test_acc = accuracy_score(yp_te, best_pipe.fit(Xp_tr, yp_tr).predict(Xp_te))
print(f"고른 스케일러: {best_name} · CV {cv_scores[best_name]:.4f} · test {test_acc:.4f}")  # 보고
print(f"2회차 KNN CV 0.7801 → {cv_scores[best_name]:.4f}")  # 숙제 결과

없음        CV 0.7801 ± 0.0355
Standard  CV 0.9854 ± 0.0136
MinMax    CV 0.9854 ± 0.0136
Robust    CV 0.9854 ± 0.0136


고른 스케일러: Standard · CV 0.9854 · test 1.0000
2회차 KNN CV 0.7801 → 0.9854
